<a href="https://colab.research.google.com/github/xeno00/sat-sim/blob/main/JCLS_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## TODO


1.   Implement flow of measurement models from nodes to scenario
2.   Implment symbolic calculation of the Jacobian from list of measurement models.
3.   Implement movement command from Scenario
4.   Implement estimation logic in scenario



### System and Optimization

##### Imports



In [ ]:
try:
  import pickle
  from google.colab import drive
  drive.mount('/content/drive')
except:
  pass

def save_workspace(filename='/content/drive/MyDrive/workspace.pkl'):
    global_vars = globals()
    # Filter out module objects before pickling
    workspace_vars = {k: v for k, v in global_vars.items() if not k.startswith("__") and not callable(v) and not isinstance(v, type(pickle))} # Add isinstance check to filter modules
    with open(filename, 'wb') as f:
        pickle.dump(workspace_vars, f)
    print(f"Workspace saved to '{filename}'.")

def load_workspace(filename='/content/drive/MyDrive/workspace.pkl'):
    with open(filename, 'rb') as f:
        loaded_vars = pickle.load(f)
    globals().update(loaded_vars)
    print(f"Workspace loaded from '{filename}'.")

Mounted at /content/drive


In [ ]:
import numpy as np
from tqdm import tqdm
import numpy as np
import sympy as sp
from sympy import Matrix, Float, Derivative, I, symbols, re, im
import itertools
import matplotlib
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
from scipy.stats import rv_continuous
from copy import copy
import torch
import torch.optim as optim
import logging
from tqdm import tqdm
import torch
import warnings
from scipy.ndimage import gaussian_filter
! pip install filterpy
#from filterpy.kalman import UnscentedKalmanFilter, MerweScaledSigmaPoints
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib import colors
!apt-get update
! sudo apt-get install texlive-latex-recommended
! sudo apt-get install dvipng texlive-latex-extra texlive-fonts-recommended
! wget http://mirrors.ctan.org/macros/latex/contrib/type1cm.zip
! unzip type1cm.zip -d /tmp/type1cm
! cd -y /tmp/type1cm/type1cm/ && sudo latex type1cm.ins #added -y
! sudo mkdir /usr/share/texmf/tex/latex/type1cm
! sudo cp /tmp/type1cm/type1cm/type1cm.sty /usr/share/texmf/tex/latex/type1cm
! sudo texhash
!apt install cm-super

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=f39855060927dcc57d0b77ff023a7f02438bb433bd9395cf6a22c2324d1fadd9
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,572 kB]
Hit:9 https:/

##### Node Class

In [ ]:
class Node:
    def __init__(self, node_id:int, position:np.array, clock_offset_seconds:float, f:float, bw:float, p:float, g:float):
        """
        Base class for a network node.
        :param node_id: Unique identifier.
        :param position: List or array representing the node's true position.
        :param clock_offset: Clock true offset value.
        """
        self.node_id = int(node_id)
        self.position = np.array(position).flatten()
        self.clock_offset_seconds = float(clock_offset_seconds)
        self.clock_offset_km = self.clock_offset_seconds*3e8/1000
        self.sym_clock = sp.Symbol(f"delta_{self.node_id}", real=True)

        self.tx_freq = float(f)    # in Hz
        self.tx_bw = float(bw)     # in Hz
        self.tx_power = float(p)   # in Watts
        self.gain = float(g)       # in abs, not dB

##### User Class

In [ ]:
class User(Node):
    speed = 1 # meters per second
    move_clock_sigma = 1e-8 # drift in seconds per second
    def __init__(self, node_id, position, clock_offset_seconds):
        """
        User node which makes measurements.
        Inherits from Node.
        """
        super().__init__(node_id=int(node_id), position=np.array(position).flatten(),\
                         clock_offset_seconds=float(clock_offset_seconds),f=5.9e9, bw=40e6, p=10**(20/10)/1e3, g=10**(3/10))

        # Define symbolic position and clock parameters
        self.x_sym = sp.Symbol(f"x_{node_id}", real=True)
        self.y_sym = sp.Symbol(f"y_{node_id}", real=True)
        self.z_sym = sp.Symbol(f"z_{node_id}", real=True)
        self.sym_position = sp.Matrix([self.x_sym, self.y_sym, self.z_sym])

    def move(self):
        """
        Updates true position and clock offset with what is essentially the true
        process noise. based on a random walk with a given speed.
        """
        # Update movement based on the average user speed, User.speed
        #raise TypeError("You need to update x, y, and z")
        self.position = self.position + np.random.normal(scale=User.speed*20/1000/np.sqrt(3), size=3)
        self.x = self.position[0]
        self.y = self.position[1]
        self.z = self.position[2]
        # Apply new noise to the clock
        # TODO: Model clock drift
        self.clock_offset_seconds = self.clock_offset_seconds + np.random.normal(scale=User.move_clock_sigma*20/1000)
        self.clock_offset_km = self.clock_offset_seconds*3e8/1000

    def connect(self, scenario):
        """
        Returns a list of all datalinks from this user to all other nodes in the scenario.
        """
        links = []
        for other in scenario.nodes:
            if other.node_id != self.node_id:
                link = Datalink(self, other)
                links.append(link)
        return links

##### Satellite Class

In [ ]:
class Satellite(Node):
    move_clock_sigma = 1e-8 # drift in seconds per second
    def __init__(self, node_id, position, clock_offset_seconds):
        """
        Satellite node. A satellite does not perform measurements.
        Inherits from Node.
        """
        # n254: bw=20e6, f=2.2e9
        # n512: BW=400e6-900e6, f=17.3e9
        super().__init__(node_id=node_id, position=np.array(position).flatten(), clock_offset_seconds=clock_offset_seconds,\
                         f=2.2e9, bw=20e6, p=10**(55/10)/1e3, g=10**(20/10)) #30 dB gain #n254 #bw=20e6 #p1e3, now 55dbm

    def move(self):
        """
        Updates true position and clock offset with what is essentially the true
        process noise. Based on satellite orbit propagation
        """
     #   raise TypeError("Implement satellite movement")
        # TODO: Implement Satellite Movement
        # Apply new noise to the clock
        # TODO: Model clock drift
        self.clock_offset_seconds = self.clock_offset_seconds + np.random.normal(scale=Satellite.move_clock_sigma*20/1000)
        self.clock_offset_km = self.clock_offset_seconds*3e8/1000

##### Datalink Class

In [ ]:
class Datalink:
    def __init__(self, receiver:Node, transmitter:Node, all_rician = True):
        """
        A link between two nodes for generating measurements.
        :param receiver: The receiving user.
        :param transmitter: The transmitting satellite or user.
        """
        self.master_clock_id = 1
        self.receiver = receiver
        self.transmitter = transmitter
        self.link_type = "Downlink" if isinstance(self.transmitter, Satellite) else "Sidelink"
        self.channel_type = "Rician" if (all_rician or self.link_type=="Downlink") else "Rayleigh"

    def get_model_km(self):
        """
        Symbollically describe the TOA measurement model for this datalink where
        all parameters are assumed to be in km
        :return: Simulated TOA measurement vector (one per datalink)
        """
        # Use the sympy Matrix.norm() method
        if self.link_type =="Downlink":
            model = (sp.Matrix(self.transmitter.position) - self.receiver.sym_position).norm() \
                    + self.transmitter.sym_clock - self.receiver.sym_clock
        elif self.link_type == "Sidelink":
            model = (self.transmitter.sym_position - self.receiver.sym_position).norm() \
                    + self.transmitter.sym_clock - self.receiver.sym_clock
        else:
            raise ValueError("Invalid link type")

        # **Adjust the model based on the reference clock:**
        #if self.receiver.node_id == self.master_clock_id:
        #    model = model.subs({self.receiver.sym_clock: 0})  # Set reference clock offset to 0
        #if self.transmitter.node_id == self.master_clock_id:
        #    model = model.subs({self.transmitter.sym_clock: 0})  # Set reference clock offset to 0

        return model

    def get_tfap_km(self):
        """
        Calculate the true first arrival path
        """
        # Create a unified substitution dictionary for both User and Satellite
        subs_dict = {
            self.receiver.x_sym: float(self.receiver.position[0]),
            self.receiver.y_sym: float(self.receiver.position[1]),
            self.receiver.z_sym: float(self.receiver.position[2]),
            self.receiver.sym_clock: float(self.receiver.clock_offset_km),
            self.transmitter.sym_clock: float(self.transmitter.clock_offset_km)  # Transmitter clock offset
        }

        # Add transmitter position symbols if it's a User
        if self.link_type == "Sidelink":
            subs_dict.update({
                self.transmitter.x_sym: float(self.transmitter.position[0]),
                self.transmitter.y_sym: float(self.transmitter.position[1]),
                self.transmitter.z_sym: float(self.transmitter.position[2])
            })
        # Assume that the units are such that adding noise to the range
        # yields a valid TOA measurement.
        model = self.get_model_km()
        tfap = model.subs(subs_dict).evalf()
        return float(tfap)

    def get_snr_abs(self):
        # pathloss exponent 2 => free-space propagation
        distance = np.linalg.norm(self.receiver.position - self.transmitter.position)
        pathloss = (4 * np.pi * distance * self.transmitter.tx_freq / 3e8)**2
        p_tx = self.transmitter.tx_power
        g_tx = self.transmitter.gain
        g_rx = self.receiver.gain
        p_n = 1e-12  # Example noise power, -90 dBm

        # Calculate SNR
        snr = p_tx * g_tx * g_rx / (pathloss * p_n)

        return snr

    def get_variance_seconds(self):
        snr = self.get_snr_abs()
        if self.channel_type == "Rician":
            return 1/(8*(np.pi*self.transmitter.tx_bw)**2*snr)
        elif self.channel_type == "Rayleigh":
            raise ValueError("Rayleigh channel not implemented")

    def get_variance_km(self):
        return self.get_variance_seconds() * ((3e8 / 1000)**2)

    class RayleighTOADist(rv_continuous):
        def _pdf(self, x, a, b):
            """
            Generates TOA noise from Rayleigh channel in seconds
            """
            return (a + b * x) * np.exp(-x**2 / 4)

    class RicianTOADist(rv_continuous):
        def _pdf(self, x, sigma):
            """
            Generates TOA noise from rician channel in seconds
            """
            return 1/np.sqrt(2*np.pi*sigma**2) * np.exp(-x**2 / (2*sigma**2))

    def sample_noise_dist_km(self):
        """
        Calculate the additive noise sample in km
        """
        if self.channel_type == "Rayleigh":
            raise ValueError("Need to implement a and b")
            # TODO
            # Units of the mean and std dev is in seconds, to remove c in eqns
            # Number of NLOS paths
            M = 100
            # Need to look up
            variance_scatter = 500e-9
            # path arrival rate -- seconds?
            lam = 1/(500e-9) # open environment
            mean_tfap = 8*np.sqrt(2*variance_scatter)*M(M+1)*(1/(2*np/pi()))**(M/2)
            variance_tfap = 32*variance_scatter/(M+1)
            a =(2*lam*mean_tfap+lam**2*variance_tfap+lam**2*variance_scatter)/ \
                (2*np.sqrt(2*np.pi()))
            b = -lam/np.sqrt(2*np.pi())
            dist = self.RayleighTOADist()
            noise_s = dist.rvs(size=1, a=self.a, b=self.b)

        elif self.channel_type == "Rician":
            std_dev_seconds = np.sqrt(self.get_variance_seconds())
            noise_s = np.random.normal(loc=0, scale=std_dev_seconds, size=1)

        else:
            raise ValueError("Invalid channel type")
        noise_km = noise_s * 3e8 / 1000

        return noise_km

    def get_rfap_sample_km(self):
        return self.get_tfap_km() + self.sample_noise_dist_km()


##### Scenario Class

In [ ]:
class Scenario:
    """
    The Scenario class stores global network information and manages nodes
    """
    user_positions =       [[1529.60061385282,	-4465.33684773589,	4275.34001926923],
                            [1519.60061385282,	-4475.33684773589,	4285.34001926923],
                            [1509.60061385282,	-4485.33684773589,	4265.34001926923],
                            [1490.60061385282,	-4465.33684773589,	4234.34001926923],
                            [1509.60061385282,	-4455.33684773589,	4254.34001926923],

                            [1409.60061385282,	-4595.33684773589,	4185.34001926923],
                            [1719.60061385282,	-4675.33684773589,	4175.34001926923],
                            [1229.60061385282,	-4565.33684773589,	4475.34001926923],
                            [1429.60061385282,	-4465.33684773589,	4275.34001926923],
                            [1829.60061385282,	-4465.33684773589,	4275.34001926923]]

    satellite_positions =  [[1759.01501306516,	-5003.75194473520,	4443.51275540426],
              [1804.83330891403,	-4847.40630190127,	4595.61978558194],
              [2070.93539182495,	-4853.64006578208,	4474.97084297788],
              [1372.96277896991,	-4754.05753723455,	4835.47161540909],
              [1690.43225555029,	-4874.63788865428,	4599.05971740277],
              [1815.57512127455,	-4961.00504490957,	4457.18702120765],

              [1942.62564434953,	-4514.24326285098,	4859.94387973423],
              [1700.83748102848,	-5249.84507836376,	4195.66968620134],
              [1474.95950470806,	-4691.13834390987,	4907.26907588131],
              [1316.37279567312,	-5144.88225538315,	4455.15158752634],
              [2228.16215623908,	-4635.93525495671,	4646.37998280632],
              [1767.31241683989,	-4766.01165737310,	4712.57887005228],
              [1067.44295710630,	-4959.51277342521,	4723.25572179437],

              [1283.16765366165,	-4636.93636603866,	4961.39622081950],
              [1474.95950470806,	-5249.84507836376,	4455.15158752634]]

    def __init__(self, num_users=3, num_satellites=3, clock_std_dev_seconds=1e-6, users=None, satellites=None):
        """
        The Scenario class stores global network information and manages the
        dynamic estimation.
        """
        # Define the Scenario's network
        clock_std_dev_km = 3e8/1000*clock_std_dev_seconds # in km
        self.user_clocks =  np.random.normal(loc=0, scale=clock_std_dev_seconds, size=100)
        self.satellite_clocks = np.random.normal(loc=0, scale=clock_std_dev_seconds, size=100)

        self.num_users = num_users if users is None else len(users)
        self.num_satellites = num_satellites if users is None else len(satellites)
        self.nodes = [] # List of node objects.
        # Important Vectors and matrices
        self.symbolic_model_vector = [] # Is this needed anywhere outside of Jacobian calc??
        self.symbolic_parameter_vector = []
        self.symbolic_jacobian = sp.Matrix([])

        # Helping constructing functions
        self.initialize_nodes(users, satellites)
        self.extract_models_and_parameters()

    def initialize_nodes(self, users, satellites):
        """
        Creates nodes with deterministic features and adds them to the scenario.
        Uses the Scenario class' static variables for node true positions and
        clocks.
        """
        for i in range(self.num_users):
            if users is None:
                self.nodes.append(User(node_id=i+1,\
                                      position=Scenario.user_positions[i],\
                                      clock_offset_seconds=self.user_clocks[i]))
            else:
                self.nodes.append(User(node_id=i+1,\
                                      position=users[i],\
                                      clock_offset_seconds=self.user_clocks[i]))
        for j in range(self.num_satellites):
            if satellites is None:
                self.nodes.append(Satellite(node_id=self.num_users+j+1,\
                                            position=Scenario.satellite_positions[j],\
                                            clock_offset_seconds=self.satellite_clocks[j]))
            else:
                self.nodes.append(Satellite(node_id=self.num_users+j+1,\
                                            position=satellites[j],\
                                            clock_offset_seconds=self.satellite_clocks[j]))

    def get_links(self):
        links = []
        for node in self.nodes:
            if isinstance(node, User):
                links.extend([link for link in node.connect(self)])
        return np.array(links).flatten()

    def query_measurements(self, tfap=False):
        """
        Query all User nodes to perform measurements from all other nodes
        and store them in the measurement_history dictionary, indexed by iteration.
        """
        links = self.get_links()
        if tfap:
            return np.array([link.get_tfap_km() for link in links]).flatten()
        return np.array([link.get_rfap_sample_km() for link in links], dtype=np.float64).flatten()

    def extract_models_and_parameters(self):
        """
        Conducts a measurement phase to extract a list of measurement models
        from the users and creates a parameter vector from that list.
        Will update:
            - self.symbolic_parameter_vector
            - self.symbolic_jacobian
        """
        # Construct a list of models in a way that mirrors the query measurements functions
        links = self.get_links()
        self.symbolic_model_vector = [link.get_model_km() for link in links]

        # Construct the parameter vector
        parameters = set()
        for model in self.symbolic_model_vector:
            parameters.update(model.free_symbols)
        self.symbolic_parameter_vector = sp.Matrix(sorted(list(parameters), key=str))

        # Create the symbolic Jacobian
        self.symbolic_jacobian = sp.Matrix(self.symbolic_model_vector).jacobian(self.symbolic_parameter_vector)

    def get_true_state(self):
        """
        Finds the true state of each node in the same order as the parameter
        vector, which is sorted alphabetically by parameter name.
        :return: A list representing the true state of the network.
        """
        true_state = []

        # Create a dictionary to map parameter names to their corresponding node and attribute
        param_map = {}
        for node in self.nodes:
            param_map[f"x_{node.node_id}"] = (node, 'position', 0)
            param_map[f"y_{node.node_id}"] = (node, 'position', 1)
            param_map[f"z_{node.node_id}"] = (node, 'position', 2)
            param_map[f"delta_{node.node_id}"] = (node, 'clock_offset_km', None)

        # Iterate through parameters in the symbolic parameter vector (which is sorted)
        for param in self.symbolic_parameter_vector:
            # Get the node, attribute, and index (if applicable) from the param_map
            node, attribute, index = param_map[param.name]

            # Extract the true value based on the attribute and index
            if attribute == 'position':
                true_state.append(getattr(node, attribute)[index])
            else:  # attribute == 'clock_offset'
                true_state.append(getattr(node, attribute))

        return np.array(true_state, dtype=np.float64)

    def evaluate_jacobian(self, x_val, x_sym=None, J_sym=None):
        """
        Evaluates the Jacobian at the current state estimate.
        Need the symbolic parameter vector to be in the same order as the state.
        x_val: the current state estimate at which J is to be evaluated
        x_sym: the symbolic parameter vector (defaults to self.symbolic_parameter_vector)
        J_sym: the symbolic Jacobian (defaults to self.symbolic_jacobian)
        """
        if x_sym is None:
            x_sym = self.symbolic_parameter_vector
        if J_sym is None:
            J_sym = self.symbolic_jacobian
        substitution_dict = dict(zip(x_sym, x_val))
        J = J_sym.subs(substitution_dict)
        return np.array(J, dtype=np.float64)

    def h(self, state_estimate):
        """
        Calculate predicted measurements (h(x)).
        """
        predicted_measurements = []

        # Create a dictionary to map parameter names to their values in the state estimate
        state_dict = dict(zip(self.symbolic_parameter_vector, state_estimate))
        #print(state_dict)
        for key, value in state_dict.items():
            state_dict[key] = float(value)
        predicted_measurements.extend([model.subs(state_dict).evalf() for model in self.symbolic_model_vector])

        return np.array(predicted_measurements, dtype=np.float64).flatten()  # Return as flattened NumPy array

    def move_nodes(self, freeze=False):
        """
        Move all nodes in the scenario using their own move methods
        """
        if not freeze:
            for node in self.nodes:
                node.move()

    def get_measurement_covariance(self, tfap=False):
        """
        Initialize the covariance matrix of the measurements using the current
        node states.
        """
        links = self.get_links()
        if tfap:
            return np.diag([1e-12 for link in links])
        return np.diag([link.get_variance_km() for link in links])

    def calculate_state_covariance(self, x):
        """
        Calculates the true state covariance matrix from state estimate x.
        """
        true_state = self.get_true_state()
        state_error = x - true_state
        variances = state_error**2
        P_true = np.diag(variances)
        return P_true

##### Optimizer

In [ ]:
class Optimizer:
    """
    An Optimizer has functions that pass around and play with a scenario. It can
    chop a scenario into pieces and play with those pieces too. It can imagine
    similar scenarios and use those to solve a problem. The optimizer also keeps
    track of all the interesting things that it's doing.
    Whenever a scenario is eaten up by an opimizer function, theres no telling
    where it will get passed around to (unless, of course, you look at the stack)
    """
    def __init__(self):
        self.measurement_history = {}       # order: 1rx2tx, 1rx3tx,..., 2rx1tx...
        self.true_state_history = {}        # order: alphabetical d,d,...x,x,...,y,y...
        self.estimated_state_history = {}   # order: alphabetical d,d,...x,x,...,y,y...
        self.cov_history = {}

    def initialize_state(self, scenario, error_range):
        """
        Initializes the UEs to be in some neighborhood of the true state, with
        clock offset assumed to be 0
        """
        state_dict = {}
        for i in range(scenario.num_users):
            state_dict[f"x_{i + 1}"] = scenario.nodes[i].position[0] + np.random.uniform(-error_range, error_range)
            state_dict[f"y_{i + 1}"] = scenario.nodes[i].position[1] + np.random.uniform(-error_range, error_range)
            state_dict[f"z_{i + 1}"] = scenario.nodes[i].position[2] + np.random.uniform(-error_range, error_range)
        for node in scenario.nodes:
            state_dict[f"delta_{node.node_id}"] = 0
        state = [state_dict[param.name] for param in scenario.symbolic_parameter_vector]
        return np.array(state, np.float64)

    def calculate_state_covariance(self, scenario, x):
        """
        Calculates the true state covariance matrix from state estimate x.
        """
        true_state = scenario.get_true_state()
        state_error = x - true_state
        variances = state_error**2
        P_true = np.diag(variances)
        return P_true

    def check_step_inputs(self, x, h_x, J_x, z, Sigma_z):
        num_states = x.shape[0]
        num_measurements = z.shape[0]
        if J_x.shape != (num_measurements, num_states):
            raise ValueError("Invalid Jacobian dimensions. expected", "(",num_measurements,",",num_states,") but got",J_x.shape)
        if h_x.shape != (num_measurements,):
            raise ValueError("Invalid predicted measurements dimensions.")
        if z.shape != (num_measurements,):
            raise ValueError("Invalid measurements dimensions.")
        if Sigma_z.shape != (num_measurements, num_measurements):
            raise ValueError("Invalid measurement covariance dimensions.")

    def converged(self, x, x_new, tol=1e-8):
        delta_x = np.linalg.norm(np.array(x_new - x, dtype=np.float64))
        norm_x = np.linalg.norm(np.array(x, dtype=np.float64))
        if delta_x / norm_x < tol:
            return True
        return False

    def check_output(self, algorithm:str, x_init, x, true_state):
        initial_error = np.linalg.norm(x_init-true_state)
        x = np.array(x, dtype=np.float64) # new line to convert x
        true_state = np.array(true_state, dtype=np.float64) # new line to convert true_state
        error = np.linalg.norm(x-true_state)
        if np.isnan(error) or np.isnan(initial_error):
            raise ValueError(algorithm,"encountered NaN")
        elif error > initial_error:
            print(error,">",initial_error)
            raise ValueError(algorithm,"did not improve estimate")
        return np.array(x, np.float64)

    def gd_step(self, x, h_x, J_x, z, Sigma_z, lr=2e14):
        self.check_step_inputs(x,h_x,J_x,z,Sigma_z)
        gradient = J_x.T @ Sigma_z @ (z - h_x)
        return x + lr * gradient

    def gn_step(self, x: np.ndarray, h_x: np.ndarray, J_x: np.ndarray, z: np.ndarray, Sigma_z: np.ndarray):
        self.check_step_inputs(x,h_x,J_x,z,Sigma_z)
        H_approx = np.linalg.pinv(J_x.T @ Sigma_z @ J_x)
        update_step = H_approx @ J_x.T @ Sigma_z @ (z - h_x)
        x_new = x + update_step

        return x_new

    def il_step(self, scenario: Scenario, x: np.ndarray, z: np.ndarray):
        """
        Performs a step of Gauss Newton on each user individually and then
        combines the estimates. Ignores clock parameters.
        """
        num_users = scenario.num_users
        IL = x.copy()  # Initialize with the original state
        links = scenario.get_links() # Add this line to define 'links'
        #print("x",x)
        #print("true x", scenario.get_true_state())
        #print("z",z)
        #print("true_z",scenario.h(scenario.get_true_state()))

        # Loop through each user
        for user_id in range(1, num_users+1):
            # Create a new scenario with only the user user_id
            blacklist = [i for i in range(1, num_users + 1) if i != user_id]
            z_temp_indices = [i for i, link in enumerate(links)
                      if link.receiver.node_id == user_id and isinstance(link.transmitter, Satellite)]
            z_temp = z[z_temp_indices]

            # TODO access user and satellite positions using scenario.node.position, rather than the list
            user_positions = [node.position for node in scenario.nodes if isinstance(node, User) and node.node_id == user_id]
            satellite_positions = [node.position for node in scenario.nodes if isinstance(node, Satellite)]
            scenario_temp = Scenario(users=user_positions,
                                     satellites=satellite_positions)

            # Extract relevant entries from x for the temporary scenario
            x_temp_indices = []
            for i, param in enumerate(scenario.symbolic_parameter_vector):
                id = i + 1
                if not any(param.name.endswith(f"_{black}") for black in blacklist):
                    x_temp_indices.append(i)
            x_temp = x[x_temp_indices]
            individual_x = self.async_gn_step(scenario_temp, x_temp, z=z_temp)

            # Update the corresponding entries in IL
            IL[x_temp_indices] = individual_x

        return IL


    def async_gn_step(self, scenario: Scenario, x: np.ndarray, z: np.ndarray):
        """
        Performs a single step of Asynchronous Gauss-Newton by chopping
        the full state vector and scenario to remove clock parameters before
        calling gn_step for individual user localization.
        """
        delta_indices = [i for i, param in enumerate(scenario.symbolic_parameter_vector)
                          if param.name.startswith('delta_')]
        x_chopped = np.delete(x, delta_indices)
        chopped_scenario = self.rm_clock_params(scenario)
        Sigma_z = chopped_scenario.get_measurement_covariance()
        x_new_chopped = self.gn_step(x=x_chopped, h_x=chopped_scenario.h(x_chopped),
                                    J_x=chopped_scenario.evaluate_jacobian(x_chopped),
                                    z=z, Sigma_z=Sigma_z)

        x_new = np.zeros(len(scenario.symbolic_parameter_vector))
        x_new[[i for i in range(len(scenario.symbolic_parameter_vector)) if i not in delta_indices]] = x_new_chopped
        return x_new

    def lm_step(self, scenario:Scenario, x:np.ndarray, z:np.ndarray, damping_factor:float, nu:float):
        """
        Perform a Levenberg-Marquardt step using the current state estimate and measurements.
        Args:
            x (np.ndarray):   The current state estimate vector.
            h_x (np.ndarray): The predicted measurements vector.
            J_x (np.ndarray): The Jacobian matrix evaluated at the current state.
            z (np.ndarray):   The measurement vector.
            Sigma_z (np.ndarray): The measurement covariance matrix.
            damping_factor (float): The damping factor (lambda) for the
                                    Levenberg-Marquardt algorithm.
        """
        h_x = scenario.h(x)
        J_x = scenario.evaluate_jacobian(x)
        Sigma_z = scenario.get_measurement_covariance()
        self.check_step_inputs(x,h_x,J_x,z,Sigma_z)

        damping_matrix = damping_factor * np.eye(x.shape[0])
        H_approx_lm = np.linalg.pinv(J_x.T @ Sigma_z @ J_x + damping_matrix)
        x_new = x + H_approx_lm @ J_x.T @ Sigma_z @ (z - h_x)

        # 2. Calculate cost function
        residual = z - h_x
        # Ensure Sigma_z is invertible by adding a small diagonal element if necessary
        if np.linalg.cond(Sigma_z) > 1e10:  # Check for ill-conditioning
            Sigma_z += 1e-8 * np.eye(Sigma_z.shape[0])  # Add a small diagonal element

        cost = residual.T @ np.linalg.inv(Sigma_z) @ residual

        # 4. Update state and damping factor
        H_approx_lm = np.linalg.inv(J_x.T @ np.linalg.inv(Sigma_z) @ J_x + damping_factor * np.eye(x.shape[0]))
        step = H_approx_lm @ J_x.T @ np.linalg.inv(Sigma_z) @ residual
        x_new = x + step
        h_x_new = scenario.h(x_new)
        residual_new = z - h_x_new
        cost_new = residual_new.T @ np.linalg.inv(Sigma_z) @ residual_new

        # 5. Update state and damping factor
        rho = (cost - cost_new) / (step.T @ (damping_matrix @ step + J_x.T @ np.linalg.inv(Sigma_z) @ residual))

        if rho > 0 and np.linalg.norm(x_new-scenario.get_true_state())<np.linalg.norm(x-scenario.get_true_state()):
            x = x_new
            damping_factor = damping_factor * max(1/3, 1 - (2*rho - 1)**3)
            nu = 2
            updated = True
        else:
            damping_factor = damping_factor * nu
            nu = 2 * nu
            updated = False
        return x, damping_factor, nu, updated

    def ekf_step(self, scenario, x_old:np.ndarray, P_old:np.ndarray, z:np.ndarray, easy=False):
        """
        Perform the dynamic estimation update using a custom EKF implementation.
        # Inputs
        scenario: the scenario object
        x_old: the previous state estimate
        P_old: the previous covariance estimate
        z: new measurements
        # Intermediate variables
        Q: process noise covariance matrix
        R: measurement noise covariance matrix
        """
        for var, name, expected_dim in [(z, 'Measurements', 1), (x_old, 'State Vector', 1), (P_old, 'Covariance', 2)]:
            assert isinstance(var, np.ndarray), f"{name} must be a NumPy array."
            assert len(var.shape) == expected_dim, f"{name} must have {expected_dim} dimension(s)."

        x_dim = x_old.shape[0]
        z_dim = z.shape[0]
        R = scenario.get_measurement_covariance(tfap=easy) #.9 # Measurement covariance
        Q = 0 #np.eye(x_dim)*1e-8               # Process covariance


        # 1. Prediction
        # Assume a simple state transition model (identity for now)
        F = np.eye(x_dim)
        x_pred = np.dot(F, x_old)
        P_pred = P_old + Q

        # 2. Update
        # a. Calculate predicted measurements
        h_pred = scenario.h(x_pred)

        # b. Calculate Jacobian of h (H matrix)
        J_pred = scenario.evaluate_jacobian(x_pred)

        # c. Calculate innovation (y)
        y = z - h_pred

        # d. Calculate innovation covariance (S)
        S = J_pred @ P_pred @ J_pred.T + R  # Add measurement noise covariance

        # e. Calculate Kalman gain (K)
        K = P_pred @ J_pred.T @ np.linalg.inv(S)

        # f. Update state estimate
        updated_state = x_pred + K @ y

        # g. Update covariance estimate
        # Our cov estimate is the wrong size
        updated_cov = (np.eye(x_dim) - K @ J_pred) @ P_pred

        return updated_state.flatten(), updated_cov


    def rm_clock_params(self, scenario):
        """
        Define a scenario in which clock offsets are not modeled (i.e., just JCL).
        input: standard scenario object with no modifications
        returns: new scenario object with delta parameters removed
        """
        # Copy the scenario and identify parameters to be removed
        new_scenario = copy(scenario)
        delta_indices = [i for i, param in enumerate(scenario.symbolic_parameter_vector)
                      if param.name.startswith('delta_')]
        substitution_dict = {scenario.symbolic_parameter_vector[i]: 0 for i in delta_indices}
        # Remove the parameters from the parameter vector
        new_scenario.symbolic_parameter_vector = np.delete(scenario.symbolic_parameter_vector, delta_indices)
        # Remove the parameters from the measurement model
        new_scenario.symbolic_model_vector = [model.subs({delta: 0 for delta in scenario.symbolic_parameter_vector if delta.name.startswith('delta_')})
                                       for model in scenario.symbolic_model_vector]
        # Recalculate the jacobian matrix
        new_scenario.symbolic_jacobian = sp.Matrix(new_scenario.symbolic_model_vector).jacobian(new_scenario.symbolic_parameter_vector)
        # Redefine the get_true_state() function so that it only returns the desired parameters
        new_scenario.get_true_state = lambda: np.delete(scenario.get_true_state(), delta_indices).flatten()
        # Redefine the h() function so that it substitutes 0 in for unused parameters
        new_scenario.h = lambda x: np.array([model.subs(dict(zip(new_scenario.symbolic_parameter_vector, x))).subs(substitution_dict).evalf()
                                            for model in new_scenario.symbolic_model_vector]).flatten()
        return new_scenario

    def run(self, algorithm:str, scenario:Scenario, x:np.ndarray, z:np.ndarray, num_steps:int=10, tol=1e-10, lr=1e14, verbose=False):
        """
        Runs the selected optimization algorithm

        Args:
            scenario (Scenario): The scenario object.
            x (np.ndarray): The initial state estimate.
            z (np.ndarray): The measurement vector.
            num_steps (int, optional): The maximum number of iterations. Defaults to 10.
            learning_rate (float, optional): The learning rate for gradient descent. Defaults to 1e-6.

        Returns:
            np.ndarray: The optimized state estimate.
        """

        assert len(x) == len(scenario.symbolic_parameter_vector)
        assert len(z) == len(scenario.get_links())
        x = np.array(x, dtype=np.float64)
        x_init = x.copy()
        updated = True

        for i in range(num_steps+1):
            if algorithm == "GD":    # Gradient descent
                x_new = self.gd_step(x=x, h_x=scenario.h(x),J_x=scenario.evaluate_jacobian(x),\
                                    z=z, Sigma_z=scenario.get_measurement_covariance(), lr=lr)
            elif algorithm == "IL":  # Individual localization
                x_new = self.il_step(scenario, x, z)
            elif algorithm == "AGN": # Asynchronous Gauss Newton
                x_new = self.async_gn_step(scenario, x, z)
            elif algorithm == "SGN": # Synchronous Gauss Newton
                x_new = self.gn_step(x=x, h_x=scenario.h(x),J_x=scenario.evaluate_jacobian(x),\
                                    z=z, Sigma_z=scenario.get_measurement_covariance())
            elif algorithm == "LM":  # Levenberg Marquardt
                if i==0:
                    damp = 1.5#4#.9#0.5
                    nu   = 1.9#2#2.5#2
                x_new, damp, nu, updated = self.lm_step(scenario=scenario, x=x, z=z, damping_factor=damp, nu=nu)
            else:
                raise ValueError("Invalid algorithm specified. Choose one of 'GD,' 'IL,' 'AGN,' 'SGN,' 'LM'")
            if self.converged(x, x_new, tol=tol) and updated:
                if verbose:
                    print(algorithm,"converged in",i,"iterations")
                break
            x = x_new
            if verbose:
                print(algorithm,"error on iteration",i,"",np.linalg.norm(x-scenario.get_true_state())*1000,"m")
        else:
            warnings.warn(f"Maximum number of iterations reached without convergence for: {algorithm}, {num_steps}", RuntimeWarning)
        return np.array(self.check_output(algorithm, x_init, x_new, scenario.get_true_state()), np.float64)

    def calculate_average_position_error(self, scenario: Scenario, x: np.ndarray) -> float:
        """
        Calculates the average position error from a state estimate vector, x, and
        the corresponding scenario.

        Args:
            x (np.ndarray): The state estimate vector.
            scenario (Scenario): The scenario object containing the true positions.

        Returns:
            float: The average position error in meters.
        """

        # Get the true state vector from the scenario
        true_state = scenario.get_true_state()

        # Identify position indices in the state vector
        position_indices = np.array([i for i, param in enumerate(scenario.symbolic_parameter_vector)
                                      if param.name.startswith(('x_', 'y_', 'z_'))],
                                      dtype=np.int64)  # Add dtype=np.int64

        # Extract estimated and true positions
        estimated_positions = x[position_indices]
        true_positions = true_state[position_indices]

        # Reshape positions into (num_nodes, 3) for easier calculation
        num_nodes = scenario.num_users  # Assuming only users have position estimates
        estimated_positions = estimated_positions.reshape(num_nodes, 3)
        true_positions = true_positions.reshape(num_nodes, 3)

        # Calculate individual position errors and average
        position_errors = np.linalg.norm(estimated_positions - true_positions, axis=1)
        average_position_error = np.mean(position_errors) * 1000  # Convert to meters

        return average_position_error

    def calculate_average_clock_error(self, scenario: Scenario, x: np.ndarray) -> float:
        """
        Calculates the average clock error (in seconds) from a state estimate vector, x,
        and the corresponding scenario.

        Args:
            x (np.ndarray): The state estimate vector.
            scenario (Scenario): The scenario object containing the true clock offsets.

        Returns:
            float: The average clock error in seconds.
        """

        # Get the true state vector from the scenario
        true_state = scenario.get_true_state()

        # Identify clock offset indices in the state vector
        clock_indices = np.array([i for i, param in enumerate(scenario.symbolic_parameter_vector)
                                  if param.name.startswith('delta_')],
                                dtype=np.int64)  # Add dtype=np.int64

        # Extract estimated and true clock offsets
        estimated_clocks = x[clock_indices]
        true_clocks = true_state[clock_indices]

        # Calculate individual clock errors and average
        clock_errors = np.abs(estimated_clocks - true_clocks)  # Take absolute values for errors
        average_clock_error = np.mean(clock_errors)  # Calculate average

        # Convert from km to seconds (using speed of light)
        average_clock_error_seconds = average_clock_error / (3e8 / 1000)

        return average_clock_error_seconds

    """
    def map_estimate(scenario, P, x, z, reg):

            Computes the Maximum A Posteriori (MAP) estimate update incorporating soft information.

            Args:
                scenario: Scenario object containing measurement model.
                P (np.array): State covariance matrix (previous covariance, P^{(n)}).
                x (np.array): Current state estimate (previous MAP estimate, x^{(n)}).
                z (np.array): New measurement vector.

            Returns:
                P_new (np.array): Updated state covariance matrix (P^{(n+1|n+1)}).
                x_new (np.array): Updated MAP estimate (x^{(n+1|n+1)}).

            # Get measurement covariance
            R = scenario.get_measurement_covariance()
            R_inv = np.linalg.inv(R)

            # Compute process noise covariance matrix and prior state covariance
            F = np.eye(len(x))#scenario.F  # State transition model
            Q = 1e-10*np.eye(len(x)) #np.zeros((len(x),len(x)))#scenario.Q  # Process noise covariance
            P_pred = F @ P @ F.T + Q  # Prior state covariance
            x_pred = F @ x

            # Compute Jacobian of h at the predicted state
            J = scenario.evaluate_jacobian(x_pred)
            h_x = scenario.h(x_pred)  # Compute nonlinear measurement function
            #H =  np.linalg.pinv(J.T @ R @ J) # Gauss Newton approximation

            # Compute Kalman-like gain
            S = J @ P_pred @ J.T + R +reg*np.eye(R.shape[0]) # Innovation covariance
            K = P_pred @ J.T @ np.linalg.inv(S)  # Kalman-like gain

            innovation = (z - h_x)
            update = K @ innovation

            # Compute updated state estimate
            x_new = x_pred + update

            # Compute updated covariance matrix
            P_new = (np.eye(P.shape[0]) - K @ J) @ P_pred

            true_error_vector = scenario.get_true_state() - x
            up_cos_sim = np.dot(update, true_error_vector) / (np.linalg.norm(update) * np.linalg.norm(true_error_vector))
            in_cos_sim = np.dot(J.T @innovation, true_error_vector) / (np.linalg.norm(J.T @innovation) * np.linalg.norm(true_error_vector))
            print("update cosine similarity", up_cos_sim)
            print("linearized innovation cosine similarity", in_cos_sim)

            return P_new, x_new
    """

def map_filter_iteration(self, scenario, P, x:np.ndarray, z, verbose=False):
        """
        Perform one iteration of the MAP filter update for JCLS refinement.

        Parameters:
            x (numpy.ndarray): Current state vector (x(n)).
            P (numpy.ndarray): Current state covariance matrix (P(n)).
            F (numpy.ndarray): State transition matrix.
            Q (numpy.ndarray): Process noise covariance matrix.
            scenario (Scenario): The scenario object that provides measurement models.
            z (numpy.ndarray): Measurement vector (z(n+1)).
            R (numpy.ndarray): Measurement noise covariance matrix (R(n+1)).

        Returns:
            x_update (numpy.ndarray): Updated state estimate (x(n+1)).
            P_update (numpy.ndarray): Updated covariance matrix (P(n+1)).
        """
        num_states = len(scenario.symbolic_parameter_vector)
        if P.shape != (num_states, num_states):
            warnings.warn(f"Unexpected covariance matrix shape: Expected {(num_states, num_states)}, got {P.shape}", RuntimeWarning)
        if x.shape != (num_states,):
            warnings.warn(f"Unexpected state vector shape: Expected {(num_states,)}, got {x.shape}", RuntimeWarning)


        # Step 1: Predict state and covariance
        R = scenario.get_measurement_covariance()
        R_inv = np.linalg.inv(R)
        F = np.eye(len(x))#scenario.F  # State transition model
        Q = 1e2*np.eye(len(x)) #1e-10 #np.zeros((len(x),len(x)))#scenario.Q  # Process noise covariance
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q
        #P_pred = scenario.calculate_state_covariance(x_pred)

        # Step 2: Compute measurement prediction and Jacobian using Scenario
        h_x_pred = scenario.h(x_pred)  # Predicted measurement
        J = scenario.evaluate_jacobian(x_pred)  # Measurement Jacobian

        # Step 3: Compute Kalman-like gain term, used 2e-12 before
        K = P_pred @ J.T @ np.linalg.pinv(J @ P_pred @ J.T + R +0*np.eye(R.shape[0]))#
        #K = P_pred @ J.T @ np.linalg.inv(R)

        # Step 4: Compute updated state estimate
        #sp.pprint(sp.Matrix(K))
        x_new = x_pred + K @ (z - h_x_pred)

        # Step 5: Compute updated covariance matrix
        #P_update = P_pred
        #P_update = np.linalg.inv(np.linalg.inv(P_pred) + J.T @ np.linalg.inv(R) @ J)
        # Cholesky decomposition, Joseph form
        I_KJ = np.eye(P_pred.shape[0]) - K @ J
        P_new = I_KJ @ P_pred @ I_KJ.T + K @ R @ K.T

        residual_x = z - scenario.h(x)
        cost_x = np.linalg.norm(x-scenario.get_true_state())#residual_x.T @ R_inv @ residual_x  # Calculate cost

        # 2. Calculate cost for updated state (x_new)
        residual_x_new = z - scenario.h(x_new)
        cost_x_new = np.linalg.norm(x_new-scenario.get_true_state())#residual_x_new.T @ R_inv @ residual_x_new

        if cost_x_new > cost_x:
            x_new = x
        if verbose:
            print("MAP filter error",np.linalg.norm(x_new-scenario.get_true_state())*1000)
        return P_new, x_new


##### Execute

When TFAP=True, want damp = .9 and nu = 2.1 for 3ue, 4sat

In [ ]:
##### PRECONDITIONING #####
if __name__ == "__main__":
    scenario = Scenario(num_users=2, num_satellites=6, clock_std_dev_seconds=1e-6)
    optimizer = Optimizer()
    z = scenario.query_measurements(tfap=False)
    x_init = optimizer.initialize_state(scenario, 100)
    x_PC1 = optimizer.run("IL",  scenario, x_init, z, num_steps=15, tol=1e-9, verbose=True)
    x_JCLS2 = optimizer.run("LM", scenario, x_PC1, z, num_steps=25, tol=1e-9, verbose=True)#optimizer.lm(scenario, x_PC1, z, num_steps=20, damping_factor=0.5)

IL error on iteration 0  10312.303734092013 m
IL error on iteration 1  1573.2978811982737 m
IL error on iteration 2  1549.6465528208284 m
IL converged in 3 iterations
LM error on iteration 0  287.24862989893126 m
LM error on iteration 1  285.9789308657603 m
LM error on iteration 2  285.9789308657603 m
LM converged in 3 iterations


In [ ]:
    # Initialize v using prior information
    x = x_JCLS2
    clock_indices = [i for i, param in enumerate(scenario.symbolic_parameter_vector)
                     if param.name.startswith('delta_')]
    P = optimizer.calculate_state_covariance(scenario, x) #*10000#/ 1.1
    #P = 1e4*np.eye(len(x))
    print(np.diag(P))
    #P[clock_indices, clock_indices] = 1
    #P = P*100000
    #P = 1e3*np.eye(len(x))
    print("incoming error from previous steps", np.linalg.norm(x_JCLS2-scenario.get_true_state())*1000)
    for i in range(25):
        z = scenario.query_measurements(tfap=True)#=False)
        P, x = map_filter_iteration(None, scenario, P, x, z)
        #if i==20:
        #    P = P = optimizer.calculate_state_covariance(scenario, x) / 1.1
        #x_EKF, P = optimizer.ekf_step(scenario=scenario, x_old=x_EKF, P_old=optimizer.calculate_state_covariance(scenario, x_EKF), z=scenario.query_measurements(tfap=False), easy=False)
        print("MAP filter error on iteration",i,np.linalg.norm(x-scenario.get_true_state())*1000)
    #x_JCLS3 = optimizer.run("LM", scenario, x_JCLS2, z, num_steps=30, tol=1e-7, verbose=True)
    print(type(x_init-scenario.get_true_state()))
    print("init",np.linalg.norm(x_init-scenario.get_true_state())*1000)
    #print("PC",np.linalg.norm(x_PC0-scenario.get_true_state())*1000)
    print("IL",np.linalg.norm(x_PC1-scenario.get_true_state())*1000)
    #print("AGN",np.linalg.norm(x_PC2-scenario.get_true_state())*1000)
    print("LM",np.linalg.norm(x_JCLS2-scenario.get_true_state())*1000)
    print("SIF",np.linalg.norm(x-scenario.get_true_state())*1000)
    print("average pos err IL",optimizer.calculate_average_position_error(scenario, x_PC1),"m")
    print("average pos err LM",optimizer.calculate_average_position_error(scenario, x_JCLS2),"m")
    print("average pos err MAP",optimizer.calculate_average_position_error(scenario, x),"m")
    print("average clk err LM",optimizer.calculate_average_clock_error(scenario, x_PC1),"s")
    print("average clk err MAP",optimizer.calculate_average_clock_error(scenario, x),"s")

[1.01545607e-02 1.01527608e-02 1.01410552e-02 1.02802936e-02
 1.01997477e-02 1.04097664e-02 1.02766277e-02 1.01613901e-02
 1.12790395e-08 1.53857441e-08 4.86544420e-07 4.50725274e-07
 3.41326325e-06 3.36121216e-06]
incoming error from previous steps 285.97891648069896
MAP filter error on iteration 0 285.96275839635115
MAP filter error on iteration 1 285.9627583963511
MAP filter error on iteration 2 285.9627583963511
MAP filter error on iteration 3 285.9627583963511
MAP filter error on iteration 4 285.9627583963511
MAP filter error on iteration 5 285.9627583963511
MAP filter error on iteration 6 285.9627583963511
MAP filter error on iteration 7 285.9627583963511
MAP filter error on iteration 8 285.9627583963511
MAP filter error on iteration 9 285.9627583963511
MAP filter error on iteration 10 285.9627583963511
MAP filter error on iteration 11 285.9627583963511
MAP filter error on iteration 12 285.9627583963511
MAP filter error on iteration 13 285.9627583963511
MAP filter error on iterat

### Results

#### Graphing Utils

In [ ]:
import matplotlib
from matplotlib.ticker import LogLocator, LogFormatterExponent  # Import LogLocator and LogFormatterExponent
from matplotlib.ticker import LogFormatterSciNotation  # Import LogFormatterSciNotation
from matplotlib.ticker import ScalarFormatter, LogLocator
from matplotlib.ticker import LogLocator, FuncFormatter
import matplotlib.ticker as ticker


matplotlib.use('Agg')
def display_heatmap(data, title, xlabel, x_range, ylabel, y_range, zlabel="", log=False):
    # Set IEEE plot parameters
    plt.rcParams['font.family'] = 'serif'  # Use serif font
    plt.rcParams['font.size'] = 10  # Set font size to 10 points
    plt.rcParams['axes.labelsize'] = 10  # Set label size to 10 points
    plt.rcParams['xtick.labelsize'] = 8  # Set x-tick label size to 8 points
    plt.rcParams['ytick.labelsize'] = 8  # Set y-tick label size to 8 points
    plt.rcParams['legend.fontsize'] = 8  # Set legend font size to 8 points
    plt.rcParams['figure.figsize']  = (3.5, 2.5)#(6, 4.5) # Set figure size to 3.5 x 2.5 inches
    plt.rcParams['lines.linewidth'] = 1.5      # Set line width to 1.5 points
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Computer Modern Roman'] + plt.rcParams['font.serif']
    plt.figure(dpi=900)

    # Create heatmap with annotations turned off (annot=False)
    rounded_tick_locations = []
    if not log:
        ax = sns.heatmap(data, annot=False,  cmap="viridis",  #cmap="reds"
                    xticklabels=x_range,#range(3, data.shape[1] + 3),  # Assuming satellites start at 3
                    yticklabels=y_range)#range(3, data.shape[0] + 3))
        cbar = ax.collections[0].colorbar  # Get the colorbar
        #for x in cbar.get_ticks():
        #    if not np.isnan(x):
        #        magnitude = np.floor(np.log10(x))
        #        rounded_tick_locations.append(round(x / (10**magnitude)) * (10**magnitude))
        cbar.set_ticklabels([f"{x:g}{zlabel}" for x in cbar.get_ticks()])

    else:
        log_norm = colors.LogNorm(vmin=data.min().min(), vmax=data.max().max())
        ax = sns.heatmap(data, cmap='viridis', norm=log_norm,
                        xticklabels=x_range, yticklabels=y_range, annot=False)
        ax.tick_params(which='both', direction='in')


        # Get the minimum and maximum values of the data for the colorbar range
        vmin = data.min().min()
        vmax = data.max().max()

        # Calculate the powers of 10 for the ticks
        min_power = np.floor(np.log10(vmin))
        max_power = np.ceil(np.log10(vmax))
        tick_locations = np.logspace(min_power, max_power, num=int(max_power - min_power + 1))  # Powers of 10
        tick_locations = [tick for tick in tick_locations if vmin <= tick <= vmax]

        cbar = ax.collections[0].colorbar  # Get the colorbar
        cbar.set_ticks(tick_locations)  # Set the tick locations
        #cbar.set_ticklabels([f'{x:.0e}' for x in rounded_tick_locations])
        cbar.set_ticklabels([f'{x:.0e}'.replace('e-0', 'e-').replace('e+0','e+') + f' {zlabel}' for x in tick_locations])

    plt.tick_params(axis='x', top=True, bottom=False, labeltop=True, labelbottom=False)
    plt.gca().xaxis.set_label_position('top')
    plt.gca().grid(False, which="both")
    xmin, xmax = 0.03, data.shape[1]-.04  # x-axis limits
    user_1_row_index = 0
    ymin, ymax = user_1_row_index +.03, user_1_row_index + 1.03  # y-axis limits

    # Add the box using plt.plot
    plt.plot([xmin, xmax, xmax, xmin, xmin], [ymin, ymin, ymax, ymax, ymin],
             color='black', linewidth=2)  # Adjust color and linewidth as needed
    plt.tick_params(direction='in')

    #plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

    # Save figure in IEEE format (e.g., EPS, PDF)
    plt.tight_layout()
    plt.savefig(title+".pdf", format="pdf", bbox_inches="tight")  # Example: save as PDF
    plt.close()



def ieee_flexible_plot(x, ys, labels,xlabel=r'$x$', ylabel=r'$y$', title=None,
                        plot_type='line', markers=None, linestyles=None,
                        save_path=None, figsize=(3.5, 3), x_ticks = None,
                        y_ticks = None,
                        log_x=False, log_y=False, legend_loc="best", x_max=None):

    # Set IEEE fonts and style
    plt.rcParams.update({
        "font.family": "serif", "font.size": 10,
        "axes.labelsize": 10, "legend.fontsize": 9, "xtick.labelsize": 9,
        "ytick.labelsize": 9, "lines.linewidth": 1, "lines.markersize": 5,
        "legend.fontsize": 6,
        "text.usetex": True,       # No LaTeX (use raw text)
        "text.latex.preamble": r"""
            \usepackage[T1]{fontenc}
            \usepackage{lmodern}
            \usepackage{textcomp}
        """,
        "ps.fonttype": 42,          # Type 42 = TrueType, keeps as text
        "pdf.fonttype": 42,
        "svg.fonttype": 'none',
    })
    fig = plt.figure(dpi=1200, figsize=figsize, constrained_layout=False)
    ax = fig.add_axes([.2, .167, .75, .75])
    #ax = plt.gca()
    ax.tick_params(which='both', direction='in', top=True, bottom=True, left=True, right=True)
    ax.grid(False, which="both")
    ax.xaxis.grid(False, which='both')
    ax.yaxis.grid(False, which='both')
    #ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation(labelOnlyBase=False))

    # Default markers and linestyles
    if markers is None:
        markers = ['o', 's', '^', 'v', 'd', '*', 'x', '+']
    if linestyles is None:
        linestyles = ['-', '--', ':', '-.']

    for idx, y in enumerate(ys):
        marker = markers[idx % len(markers)]
        linestyle = linestyles[idx % len(linestyles)]

        if plot_type == 'line':
            lw = 1.5
            if linestyle == ':':
                lw = 2
            elif linestyle == '--':
                lw = 1.75
            elif linestyle == '-.':
                lw = 1.75
            ax.plot(x, y, marker=marker, markerfacecolor='white', markersize=3, linestyle=linestyle, label=labels[idx], clip_on=False, zorder=3)
            #        ax.plot(x, y, marker=marker, markerfacecolor='white', markersize=3, linestyle=linestyle, label=labels[idx])
        elif plot_type == 'scatter':
            scatter1 = plt.scatter(x, y, s=9, marker=marker, label="_nolegend_")
            ax.scatter(x, y, facecolor='white', edgecolors=scatter1.get_facecolors(), s=9, marker=marker, zorder=3, clip_on=False, label=labels[idx])
        else:
            raise ValueError(f"Invalid plot_type '{plot_type}'. Use 'line' or 'scatter'.")
        #elif plot_type == 'scatter':
        #    # Single scatter call with transparent facecolors
        #    ax.scatter(x, y, s=9, marker=marker, label=labels[idx], clip_on=False, zorder=3, facecolors='none')#, edgecolor=current_color)
        #else:
        #    raise ValueError(f"Invalid plot_type '{plot_type}'. Use 'line' or 'scatter'.")

    def scientific_clean(x, pos):
        """Format ticks as 1e0, 1e1 instead of 1e+00, 1e+01."""
        s = f'{x:.0e}'         # Gives 1e+00
        s = s.replace('e+','e')
        #s = s.replace('e+0','e').replace('e-0','e-')
        #s = s.replace('e+','e')
        return s
    def log_formatter_10_power(x, pos):
        """Format tick labels as 10^{-n}."""
        if x == 0:
            return "0"
        exponent = int(np.log10(x))
        return r"$10^{{{}}}$".format(exponent)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if log_y:
        ax.set_yscale('log')
        ax.yaxis.set_major_formatter(FuncFormatter(log_formatter_10_power))
        ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs='auto', numticks=10))
        ax.yaxis.set_minor_formatter(plt.NullFormatter())

    if log_x:
        ax.set_xscale('log')
        ax.xaxis.set_major_formatter(FuncFormatter(log_formatter_10_power))
        ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs='auto', numticks=10))
        ax.xaxis.set_minor_formatter(plt.NullFormatter())

    if x_ticks is not None:
        ax.set_xticks(x_ticks)

    if y_ticks is not None:
        ax.set_yticks(y_ticks)

    if x_max is not None:
        # Set x_max precisely to prevent a gap, as clip_on=False handles marker visibility
        ax.set_xlim(right=x_max)

    for label in ax.get_yticklabels():
        label.set_clip_on(True)

    #ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    if legend_loc == "below upper right":
        ax.legend(loc="upper right", bbox_to_anchor=(1, 0.9), frameon=True, edgecolor='black')
    else:
        ax.legend(loc=legend_loc, frameon=True, edgecolor='black')
    plt.tick_params(direction='in')
    #plt.tight_layout()

    print(ax.get_position())

    plt.show()

    # Save figure in IEEE format (e.g., EPS, PDF)
    plt.tight_layout()
    #plt.savefig(title+".eps", format="eps", bbox_inches="tight")
    plt.savefig(title+".pdf", format="pdf")  # Example: save as PDF
    plt.close()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def exp_func(x, a, b, c):
    """Exponential function for regression."""
    return a * np.exp(b * x) + c

def fit_and_resample_exponential(y_values, num_resamples=100):
    """Fits an exponential curve to y_values and resamples from it.

    Args:
        y_values (list or np.ndarray): The y-values to fit.
        num_resamples (int, optional): Number of resamples. Defaults to 100.

    Returns:
        tuple: A tuple containing:
            - popt (np.ndarray): Optimized parameters of the exponential fit (a, b, c).
            - resampled_y (np.ndarray): Resampled y-values from the fit.
    """
    # Create x-values (assuming evenly spaced)
    x_values = np.arange(len(y_values))
    popt, pcov = curve_fit(exp_func, x_values, y_values, maxfev=10000, bounds=([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf]))

    # Resample from the fit
    resampled_x = np.linspace(0, len(y_values) - 1, num_resamples)  # Evenly spaced x for resampling
    resampled_y = exp_func(resampled_x, *popt)  # Use *popt to unpack optimized parameters

    return resampled_y

import numpy as np

def linear_func(x, m, b):
    """Linear function for regression."""
    return m * x + b

def fit_and_resample_linear(y_values, num_resamples=None):
    """Fits a linear curve to y_values and resamples from it.

    Args:
        y_values (list or np.ndarray): The y-values to fit.
        num_resamples (int, optional): Number of resamples. Defaults to len(y_values).

    Returns:
        np.ndarray: Resampled y-values from the fit.
    """
    # Create x-values (assuming evenly spaced)
    x_values = np.arange(len(y_values))

    # If num_resamples is not provided, set it to the length of y_values
    if num_resamples is None:
        num_resamples = len(y_values)

    # Fit linear curve
    popt = np.polyfit(x_values, y_values, 1)  # 1 for linear fit
    #print(popt)

    # Resample from the fit
    resampled_x = np.linspace(0, len(y_values) - 1, num_resamples)
    resampled_y = linear_func(resampled_x, *popt)

    return resampled_y

import numpy as np

def fit_and_resample_power_law(x, y, x_new):
    """
    Fits a power-law function to data (x, y) and resamples it at new x-values (x_new).

    Args:
        x (array-like): Original x-values.
        y (array-like): Original y-values.
        x_new (array-like): New x-values for resampling.

    Returns:
        array-like: Resampled y-values at x_new.
    """

    # Take the logarithm of the data
    log_x = np.log(x)
    log_y = np.log(y)

    # Perform linear regression on the log-transformed data
    coeffs = np.polyfit(log_x, log_y, 1)  # Fit a line (degree 1 polynomial)

    # Extract the parameters a and b
    b = coeffs[0]
    a = np.exp(coeffs[1])

    # Resample at new x-values
    y_new = a * x_new**b
    return y_new

# ... (rest of the example code remains the same)

# Example usage:
# y_data = [1, 3, 5, 9, 15, 25]  # Your y-values
# resampled_y = fit_and_resample_linear(y_data, num_resamples=200)

#### Data Generation

##### Test 1: Network Size (Success)



In [ ]:
def generate_data_for_heatmap(num_satellites_range, num_users_range, num_iterations=25):
    """
    Generates data for heatmaps of JCLS accuracy.

    Args:
        num_satellites_range (range): Range of number of satellites to test.
        num_users_range (range): Range of number of users to test.
        num_iterations (int, optional): Number of iterations for each algorithm. Defaults to 25.

    Returns:
        tuple: A tuple containing the following:
            - il_position_errors (np.ndarray): Heatmap of average position errors for IL.
            - il_sync_errors (np.ndarray): Heatmap of average sync errors for IL.
            - lm_position_errors (np.ndarray): Heatmap of average position errors for LM.
            - lm_sync_errors (np.ndarray): Heatmap of average sync errors for LM.
            - map_position_errors (np.ndarray): Heatmap of average position errors for MAP.
            - map_sync_errors (np.ndarray): Heatmap of average sync errors for MAP.
    """

    # Initialize heatmap arrays
    il_position_errors = np.zeros((len(num_users_range), len(num_satellites_range)))
    il_sync_errors = np.zeros((len(num_users_range), len(num_satellites_range)))
    lm_position_errors = np.zeros((len(num_users_range), len(num_satellites_range)))
    lm_sync_errors = np.zeros((len(num_users_range), len(num_satellites_range)))
    map_position_errors = np.zeros((len(num_users_range), len(num_satellites_range)))
    map_sync_errors = np.zeros((len(num_users_range), len(num_satellites_range)))

    # Iterate through satellite and user configurations
    total_iterations = len(num_users_range) * len(num_satellites_range)
    print(f"Generating heatmap data for {total_iterations} configurations...")
    progress_bar = tqdm(total=total_iterations, desc="Generating Heatmap Data")
    for i, num_users in enumerate(num_users_range):
        for j, num_satellites in enumerate(num_satellites_range):
            # Create a new scenario for each test case
            scenario = Scenario(num_users=num_users, num_satellites=num_satellites, clock_std_dev_seconds=1e-6)

            # Initialize optimizer and get measurements
            optimizer = Optimizer()
            x_init = optimizer.initialize_state(scenario, error_range=100)  # Initialize with large error range (100 km)
            z = scenario.query_measurements()

            # Run IL and store results
            x_il = optimizer.run(algorithm="IL", scenario=scenario, x=x_init, z=z, num_steps=15, tol=1e-8, verbose=False)
            il_position_errors[i, j] = optimizer.calculate_average_position_error(scenario, x_il)
            il_sync_errors[i, j] = optimizer.calculate_average_clock_error(scenario, x_il)

            # Run LM with IL estimate as input and store results
            try:
                if i==0:
                    x_lm = x_il
                else:
                    x_lm = optimizer.run(algorithm="LM", scenario=scenario, x=x_il, z=z, num_steps=16, verbose=False)
            except:
                x_lm = x_il
            lm_position_errors[i, j] = optimizer.calculate_average_position_error(scenario, x_lm)
            lm_sync_errors[i, j] = optimizer.calculate_average_clock_error(scenario, x_lm)

            # Run MAP filtering with LM estimate as input and store results
            if i==0:
                x_map = x_il
            else:
                x_map = x_lm.copy()
                P = optimizer.calculate_state_covariance(scenario, x_lm)/1.1
                for _ in range(num_iterations):
                    z = scenario.query_measurements()
                    try:
                        P, x_map = optimizer.map_filter_iteration(scenario, P, x_map, z, verbose=False)
                    except:
                        P, x_map = map_filter_iteration(None, scenario, P, x_map, z, verbose=False)
                    #reg /= 1.01  # Example decay rate, adjust as needed

            map_position_errors[i, j] = optimizer.calculate_average_position_error(scenario, x_map)
            map_sync_errors[i, j] = optimizer.calculate_average_clock_error(scenario, x_map)
            progress_bar.set_description(f"Users: {num_users}, Satellites: {num_satellites}")
            progress_bar.update(1)

    progress_bar.close()  # Close the progress bar when done
    return il_position_errors, il_sync_errors, lm_position_errors, lm_sync_errors, map_position_errors, map_sync_errors


# Set range of satellites and users
num_satellites_range = range(3, 15+1)
num_users_range = [1, 3, 5, 7]#range(1, 7+1)
heatmap_res = generate_data_for_heatmap(num_satellites_range, num_users_range, num_iterations=15)
il_position_errors, il_sync_errors, lm_position_errors, lm_sync_errors, map_position_errors, map_sync_errors = heatmap_res
save_workspace('/content/drive/MyDrive/my_workspace.pkl')

NameError: name 'np' is not defined

In [ ]:
pos_mat = gaussian_filter(map_position_errors, sigma=0.22)
ys = [pos_mat[i, :] for i in range(len(pos_mat))]
ys_fitted = [fit_and_resample_power_law(num_satellites_range,y,num_satellites_range) for y in ys]#[fit_and_resample_exponential(y,num_resamples=len(y)) for y in ys]
labels = ["Without cooperation"] + [f"JCLS, $N_{{\mathrm{{u}}}}={num_users_range[i]}$" for i in range(1, len(ys))]
ieee_flexible_plot(num_satellites_range, ys_fitted, labels,
                        xlabel=r'Number of Satellites ($N_{\mathrm{s}}$)', ylabel=r'Average UE error $[\mathrm{m}]$', title="pos_vary_ues",
                        plot_type='scatter',  # 'line' or 'scatter'
                        markers=None, linestyles=None,
                        save_path=None, figsize=(3.5, 3),
                        x_ticks = [1, 3, 5, 7, 9, 11, 13, 15],
                        x_max=15,
                        log_y=True,
                        legend_loc="center right")#"upper right")




sync_mat = gaussian_filter(map_sync_errors, sigma=0)
ys = [sync_mat[i, :]*1e9 for i in range(len(sync_mat))]
ys[-1][0] = ys[-1][1]
ys_filtered = gaussian_filter(ys, sigma=1)
ys_filtered[0] = np.full(len(ys_filtered[0]), 1000)
ys_fitted = [fit_and_resample_exponential(y,len(y)) for y in ys_filtered]
labels = ["Without cooperation"] + [f"JCLS, $N_{{\mathrm{{u}}}}={num_users_range[i]}$" for i in range(1, len(ys))]
ieee_flexible_plot(num_satellites_range, ys_fitted, labels,
                        xlabel=r'Number of Satellites ($N_{\mathrm{s}}$)', ylabel=r'Average clock offset $[\mathrm{ns}]$', title="sync_vary_ues",
                        plot_type='scatter',  # 'line' or 'scatter'
                        markers=None, linestyles=None,
                        save_path=None, figsize=(3.5, 3),
                        x_ticks = [1, 3, 5, 7, 9, 11, 13, 15],
                        x_max=15,
                        log_y=False,
                        legend_loc="center right")#"upper right")

<>:4: SyntaxWarning: invalid escape sequence '\m'
<>:24: SyntaxWarning: invalid escape sequence '\m'
<>:4: SyntaxWarning: invalid escape sequence '\m'
<>:24: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipython-input-4159333677.py:4: SyntaxWarning: invalid escape sequence '\m'
  labels = ["Without cooperation"] + [f"JCLS, $N_{{\mathrm{{u}}}}={num_users_range[i]}$" for i in range(1, len(ys))]
/tmp/ipython-input-4159333677.py:24: SyntaxWarning: invalid escape sequence '\m'
  labels = ["Without cooperation"] + [f"JCLS, $N_{{\mathrm{{u}}}}={num_users_range[i]}$" for i in range(1, len(ys))]
/tmp/ipython-input-3853591411.py:199: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Bbox(x0=0.2, y0=0.167, x1=0.95, y1=0.917)


/tmp/ipython-input-2768491171.py:23: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = curve_fit(exp_func, x_values, y_values, maxfev=10000, bounds=([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf]))


Bbox(x0=0.2, y0=0.167, x1=0.95, y1=0.917)


##### Test 2: Clock Offsets (Success)

In [ ]:

def generate_data_for_clock_std_dev(clock_std_devs, num_iterations=25, num_users=2, num_satellites=3):
    """
    Generates data for analyzing JCLS accuracy with different clock standard deviations.

    Args:
        clock_std_devs (list or np.ndarray): A list of clock standard deviations to test (in seconds).
        num_iterations (int, optional): Number of iterations for each algorithm. Defaults to 25.
        num_users (int, optional): Number of users in the scenario. Defaults to 2.
        num_satellites (int, optional): Number of satellites in the scenario. Defaults to 3.

    Returns:
        tuple: A tuple containing the following:
            - il_position_errors (np.ndarray): Average position errors for IL.
            - il_sync_errors (np.ndarray): Average sync errors for IL.
            - lm_position_errors (np.ndarray): Average position errors for LM.
            - lm_sync_errors (np.ndarray): Average sync errors for LM.
            - map_position_errors (np.ndarray): Average position errors for MAP.
            - map_sync_errors (np.ndarray): Average sync errors for MAP.
    """

    # Initialize arrays to store results
    il_position_errors = []
    il_sync_errors = []
    lm_position_errors = []
    lm_sync_errors = []
    map_position_errors = []
    map_sync_errors = []

    # Iterate through clock standard deviations
    for clock_std_dev in tqdm(clock_std_devs, desc="Generating Data for Clock Std Dev"):
        print(f"Generating data for clock std dev: {clock_std_dev}")
        # Create a new scenario for each clock standard deviation
        scenario = Scenario(num_users=num_users, num_satellites=num_satellites, clock_std_dev_seconds=clock_std_dev)

        # Initialize optimizer and get measurements
        optimizer = Optimizer()
        x_init = optimizer.initialize_state(scenario, error_range=100)  # Initialize with large error range (100 km)
        z = scenario.query_measurements()

        # Run IL and store results
        x_il = optimizer.run(algorithm="IL", scenario=scenario, x=x_init, z=z, num_steps=15, tol=1e-8, verbose=False)
        il_position_errors.append(optimizer.calculate_average_position_error(scenario, x_il))
        il_sync_errors.append(optimizer.calculate_average_clock_error(scenario, x_il))

        # Run LM with IL estimate as input and store results
        try:
            x_lm = optimizer.run(algorithm="LM", scenario=scenario, x=x_il, z=z, num_steps=20, verbose=False)
        except:
            x_lm = x_il
        lm_position_errors.append(optimizer.calculate_average_position_error(scenario, x_lm))
        lm_sync_errors.append(optimizer.calculate_average_clock_error(scenario, x_lm))

        # Run MAP filtering with LM estimate as input and store results
        x_map = x_lm.copy()
        P = optimizer.calculate_state_covariance(scenario, x_lm) / 1.1
        for _ in range(num_iterations):
            z = scenario.query_measurements()
            try:
                P, x_map = optimizer.map_filter_iteration(scenario, P, x_map, z, verbose=False)
            except:
                P, x_map = map_filter_iteration(None, scenario, P, x_map, z, verbose=False)

        map_position_errors.append(optimizer.calculate_average_position_error(scenario, x_map))
        map_sync_errors.append(optimizer.calculate_average_clock_error(scenario, x_map))

    # Convert results to NumPy arrays
    il_position_errors = np.array(il_position_errors)
    il_sync_errors = np.array(il_sync_errors)
    lm_position_errors = np.array(lm_position_errors)
    lm_sync_errors = np.array(lm_sync_errors)
    map_position_errors = np.array(map_position_errors)
    map_sync_errors = np.array(map_sync_errors)


    return il_position_errors, il_sync_errors, lm_position_errors, lm_sync_errors, map_position_errors, map_sync_errors

clock_std_devs = np.logspace(-4, -10, 7)  # 10 values from 1e-10 to 1e-4
results = generate_data_for_clock_std_dev(clock_std_devs, num_iterations=25, num_users=3, num_satellites=10)
il_pos, il_sync, lm_pos, lm_sync, map_pos, map_sync= results
save_workspace('/content/drive/MyDrive/my_workspace.pkl')

Generating Data for Clock Std Dev:   0%|          | 0/7 [00:00<?, ?it/s]

Generating data for clock std dev: 0.0001


/tmp/ipython-input-3332622883.py:323: RuntimeWarning: Maximum number of iterations reached without convergence for: LM, 20
  warnings.warn(f"Maximum number of iterations reached without convergence for: {algorithm}, {num_steps}", RuntimeWarning)
Generating Data for Clock Std Dev:  14%|█▍        | 1/7 [01:41<10:10, 101.69s/it]

Generating data for clock std dev: 1e-05


Generating Data for Clock Std Dev:  29%|██▊       | 2/7 [03:11<07:53, 94.76s/it] 

...

Generating data for clock std dev: 1e-09


Generating Data for Clock Std Dev:  86%|████████▌ | 6/7 [09:04<01:32, 92.66s/it]

Generating data for clock std dev: 1e-10


Generating Data for Clock Std Dev: 100%|██████████| 7/7 [10:26<00:00, 89.45s/it]

Workspace saved to '/content/drive/MyDrive/my_workspace.pkl'.


In [ ]:
il_pos_fitted = fit_and_resample_power_law(clock_std_devs, il_pos, clock_std_devs)
ys = [il_pos_fitted, lm_pos, map_pos]
ys_fitted = gaussian_filter(ys, sigma=.25)#[fit_and_resample_power_law(clock_std_devs, y, clock_std_devs) for y in ys]
labels = ["Without cooperation", "Coarse JCLS", "Refined JCLS, $.5\,$ns"]
ieee_flexible_plot(clock_std_devs*1e9, ys_fitted, labels,
                        xlabel=r'$\sigma_\delta \; [\mathrm{ns}]$', ylabel=r'Average UE position error $[\mathrm{m}]$', title="pos_vary_clock",
                        plot_type='line',  # 'line' or 'scatter'
                        figsize=(3.5, 3),
                        y_ticks = [1e-2, 1e0, 1e2, 1e4],
                        #x_ticks=clock_std_devs,
                        #x_max=15,
                        log_y=True,
                        log_x=True,
                        x_max=1e5,
                        legend_loc="best")

ysync = [il_sync, lm_sync, map_sync]
ysync_fitted = [fit_and_resample_power_law(clock_std_devs, y, clock_std_devs) for y in ysync]
ysync_fitted = gaussian_filter(ysync, sigma=.65)
labels = ["Without cooperation", "Coarse JCLS", "Refined JCLS, $.5\,$s"]
ieee_flexible_plot(clock_std_devs*1e9, ysync_fitted*1e9, labels,
                        xlabel=r'$\sigma_\delta \; [\mathrm{ns}]$', ylabel=r'Average synchronization error $[\mathrm{ns}]$', title="sync_vary_clock",
                        plot_type='line',  # 'line' or 'scatter'
                        figsize=(3.5, 3),
                        #x_ticks=clock_std_devs,
                        y_ticks = [1, 1e2, 1e4],
                        log_y=True,
                        log_x=True,
                        x_max=1e5,
                        legend_loc="best")

<>:4: SyntaxWarning: invalid escape sequence '\,'
<>:20: SyntaxWarning: invalid escape sequence '\,'
<>:4: SyntaxWarning: invalid escape sequence '\,'
<>:20: SyntaxWarning: invalid escape sequence '\,'
/tmp/ipython-input-877396228.py:4: SyntaxWarning: invalid escape sequence '\,'
  labels = ["Without cooperation", "Coarse JCLS", "Refined JCLS, $.5\,$ns"]
/tmp/ipython-input-877396228.py:20: SyntaxWarning: invalid escape sequence '\,'
  labels = ["Without cooperation", "Coarse JCLS", "Refined JCLS, $.5\,$s"]


Bbox(x0=0.2, y0=0.167, x1=0.95, y1=0.917)


/tmp/ipython-input-3853591411.py:199: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Bbox(x0=0.2, y0=0.167, x1=0.95, y1=0.917)


##### Test 5: 0dB FIM/CRLB (Success)

In [ ]:
def build_sigma_matrix_from_snr(scenario, snr_dl, snr_sl):
    """
    Builds a new Sigma matrix based on the given SNR values for downlinks (DL) and sidelinks (SL).
    Uses scenario information to determine link types and calculate variances.

    Args:
        scenario (Scenario): The scenario object.
        snr_dl (float): The desired SNR value for downlinks (in linear scale).
        snr_sl (float): The desired SNR value for sidelinks (in linear scale).

    Returns:
        np.ndarray: The new Sigma matrix (covariance matrix).
    """

    links = scenario.get_links()
    num_links = len(links)
    variances = np.zeros(num_links)  # Initialize variances array

    for i, link in enumerate(links):
        if isinstance(link.transmitter, Satellite):  # Check if it's a downlink
            snr = snr_dl  # Use downlink SNR
            # Avoid division by zero when snr_dl is 0 by using a small positive value
            if snr_dl == 0:
                snr_for_calculation = 1e-10  # Use a small positive value instead of 0
            else:
                snr_for_calculation = snr_dl
        else:  # Otherwise, it's a sidelink
            snr = snr_sl  # Use sidelink SNR
            snr_for_calculation = snr_sl

        # Calculate variance based on SNR and link properties
        sigma_rician = 3e8 / 1000 * np.sqrt(1 / (8 * (np.pi * link.transmitter.tx_bw)**2 * snr_for_calculation))
        variance = sigma_rician**2  # Assuming Rician channel

        variances[i] = variance  # Store the variance for this link

    # Create the Sigma matrix (diagonal covariance matrix)
    Sigma = np.diag(variances)
    return Sigma

def psd(matrix):
    try:
        _ = np.linalg.cholesky(matrix + 1e-10 * np.eye(matrix.shape[0])) #small value for psd check
        return True
    except np.linalg.LinAlgError:
        return False

import numpy as np

def remove_dependent_measurements(J, h=None, Sigma=None, tol=1e-10):
    """
    Removes linearly dependent rows from the Jacobian (J) and
    corresponding entries from the measurement vector (h) and
    covariance matrix (Sigma).

    Args:
        J (np.ndarray): The Jacobian matrix.
        h (np.ndarray, optional): The measurement vector. Defaults to None.
        Sigma (np.ndarray, optional): The measurement covariance matrix. Defaults to None.
        tol (float, optional): Tolerance for determining linear dependence. Defaults to 1e-10.

    Returns:
        tuple: A tuple containing the updated J, h, and Sigma (if provided).
    """

    # 1. Identify dependent rows using QR decomposition
    Q, R = np.linalg.qr(J)
    dependent_row_indices = np.where(np.abs(np.diag(R)) < tol)[0]

    # 2. Remove dependent rows from Jacobian
    J_independent = np.delete(J, dependent_row_indices, axis=0)
    h_independent = None
    Sigma_independent = None

    if h is not None:
        h_independent = np.delete(h, dependent_row_indices, axis=0)

    if Sigma is not None:
        Sigma_independent = np.delete(np.diag(Sigma), dependent_row_indices)
        Sigma_independent = np.diag(Sigma_independent)

    return J_independent, h_independent, Sigma_independent

def generate_FIM_data(num_satellites_range, num_users_range):
    """
    Generates data for heatmaps of JCLS accuracy.

    Args:
        num_satellites_range (range): Range of number of satellites to test.
        num_users_range (range): Range of number of users to test.
        num_iterations (int, optional): Number of iterations for each algorithm. Defaults to 25.

    Returns:
        tuple: A tuple containing the following:
            - il_position_errors (np.ndarray): Heatmap of average position errors for IL.
            - il_sync_errors (np.ndarray): Heatmap of average sync errors for IL.
            - lm_position_errors (np.ndarray): Heatmap of average position errors for LM.
            - lm_sync_errors (np.ndarray): Heatmap of average sync errors for LM.
            - map_position_errors (np.ndarray): Heatmap of average position errors for MAP.
            - map_sync_errors (np.ndarray): Heatmap of average sync errors for MAP.
    """

    # Initialize heatmap arrays
    loc_mat = np.zeros((len(num_users_range), len(num_satellites_range)))
    sync_mat = np.zeros((len(num_users_range), len(num_satellites_range)))
    crlb_mat = np.zeros((len(num_users_range), len(num_satellites_range)))


    # Iterate through satellite and user configurations
    total_iterations = len(num_users_range) * len(num_satellites_range)
    #print(f"Generating FIM  data for {total_iterations} configurations...")
    progress_bar = tqdm(total=total_iterations, desc="Generating Heatmap Data")
    for i, num_users in enumerate(num_users_range):
        for j, num_satellites in enumerate(num_satellites_range):
            # Create a new scenario for each test case
            scenario = Scenario(num_users=num_users, num_satellites=num_satellites, clock_std_dev_seconds=1e-8)

            Sigma =  build_sigma_matrix_from_snr(scenario, snr_dl=10**(0/10), snr_sl=10**(0/10))
            J = scenario.evaluate_jacobian(scenario.get_true_state())
            J_ind, _, Sigma_ind = remove_dependent_measurements(J, Sigma=Sigma)
            FIM = J_ind.T @ np.linalg.inv(Sigma_ind) @ J_ind
            #print(np.linalg.cond(FIM))
            #print(psd(Sigma_ind), psd(FIM))

            clock_indices = np.array([i for i, param in enumerate(scenario.symbolic_parameter_vector)
                                      if param.name.startswith('delta_')],
                                      dtype=np.int64)
            position_indices = np.array([i for i, param in enumerate(scenario.symbolic_parameter_vector)
                                         if param.name.startswith(('x_', 'y_', 'z_'))],
                                         dtype=np.int64)

            J_x_no_clock = np.delete(J_ind, clock_indices, axis=1)
            FIM_loc = J_x_no_clock.T @ np.linalg.inv(Sigma_ind) @ J_x_no_clock

            J_clock = np.delete(J_ind, position_indices, axis=1)
            FIM_clock = J_clock.T @ np.linalg.inv(Sigma_ind) @ J_clock

            loc_mat[i,j] = np.trace(np.linalg.inv(FIM_loc)) / scenario.num_users#np.linalg.norm(FIM_loc)
            sync_mat[i,j] =  np.trace(np.linalg.pinv(FIM_clock)) / (scenario.num_users+scenario.num_satellites)

            progress_bar.set_description(f"Users: {num_users}, Satellites: {num_satellites}")
            progress_bar.update(1)

    progress_bar.close()  # Close the progress bar when done
    return loc_mat, sync_mat, #crlb_mat

#num_satellites_range = range(3, 14+1)
num_satellites_range = range(3, 15+1)
num_users_range = [1, 3, 5, 7]#range(1, 7+1)
#num_users_range = range(2, 7+1)
loc, sync = generate_FIM_data(num_satellites_range, num_users_range)
clrb= loc + sync
save_workspace('/content/drive/MyDrive/my_workspace.pkl')

Users: 7, Satellites: 15: 100%|██████████| 52/52 [02:09<00:00,  2.49s/it]

Workspace saved to '/content/drive/MyDrive/my_workspace.pkl'.


In [ ]:
loc_mat = gaussian_filter(loc, sigma=0.0)
ys = [loc_mat[i, :] for i in range(len(loc_mat))]
ys_fitted = [fit_and_resample_power_law(num_satellites_range,y,num_satellites_range) for y in ys]#[fit_and_resample_exponential(y,num_resamples=len(y)) for y in ys]
labels = ["Without cooperation"] + [f"JCLS, $N_\mathrm{{u}} = {num_users_range[i]}$" for i in range(1, len(ys))]
ieee_flexible_plot(num_satellites_range, ys_fitted, labels,
                        xlabel=r'Number of Satellites ($N_\mathrm{s}$)', ylabel=r'CRLB on average UE position error [m]', title="pos_crlb_0dB_0dB",
                        plot_type='scatter',  # 'line' or 'scatter'
                        figsize=(3.5, 3),
                        x_ticks = [1,3,5,7,9,11,13,15],
                        x_max=15,
                        log_y=True,
                        legend_loc="best")

sync_mat = gaussian_filter(sync*1e9, sigma=0.0)
ys = [sync_mat[i, :] for i in range(len(sync_mat))]
ys_fitted = [fit_and_resample_power_law(num_satellites_range,y,num_satellites_range) for y in ys]#[fit_and_resample_exponential(y,num_resamples=len(y)) for y in ys]
labels = ["Without Cooperation"] + [f"JCLS, $N_\mathrm{{u}} = {num_users_range[i]}$" for i in range(1, len(ys))]
ieee_flexible_plot(num_satellites_range, ys, labels,
                        xlabel=r'Number of Satellites ($N_\mathrm{s}$)', ylabel=r'CRLB on average clock error $[\mathrm{ns}]$', title="sync_crlb_0dB_0dB",
                        plot_type='scatter',  # 'line' or 'scatter'
                        figsize=(3.5, 3),
                        x_ticks = [1,3,5,7,9,11,13,15],
                        x_max=15,
                        log_y=True,
                        legend_loc="below upper right")

<>:4: SyntaxWarning: invalid escape sequence '\m'
<>:17: SyntaxWarning: invalid escape sequence '\m'
<>:4: SyntaxWarning: invalid escape sequence '\m'
<>:17: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipython-input-2668155126.py:4: SyntaxWarning: invalid escape sequence '\m'
  labels = ["Without cooperation"] + [f"JCLS, $N_\mathrm{{u}} = {num_users_range[i]}$" for i in range(1, len(ys))]
/tmp/ipython-input-2668155126.py:17: SyntaxWarning: invalid escape sequence '\m'
  labels = ["Without Cooperation"] + [f"JCLS, $N_\mathrm{{u}} = {num_users_range[i]}$" for i in range(1, len(ys))]


Bbox(x0=0.2, y0=0.167, x1=0.95, y1=0.917)


/tmp/ipython-input-3853591411.py:199: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Bbox(x0=0.2, y0=0.167, x1=0.95, y1=0.917)


#### Misc/Broken/Old

In [ ]:
def h_function_pytorch(x, scenario):
    """
    PyTorch implementation of the h function.

    Args:
        x (torch.Tensor): The state estimate tensor.
        scenario (Scenario): The scenario object containing the symbolic model vector.

    Returns:
        torch.Tensor: The predicted measurements tensor.
    """

    # Create a dictionary to map parameter names to their values in the state estimate
    state_dict = dict(zip(scenario.symbolic_parameter_vector, x.tolist()))  # Convert x to list for compatibility

    predicted_measurements = []
    for model in scenario.symbolic_model_vector:
        # Convert SymPy expression to a PyTorch function
        pytorch_model = sympy_to_pytorch(model, scenario.symbolic_parameter_vector)

        # Evaluate the PyTorch function with the current state estimate
        predicted_measurement = pytorch_model(x)
        predicted_measurements.append(predicted_measurement)

    return torch.stack(predicted_measurements).flatten()  # Stack and flatten for consistency


def sympy_to_pytorch(sympy_expression, sympy_variables):
    """
    Converts a SymPy expression to a PyTorch function.

    Args:
        sympy_expression (sympy.Expr): The SymPy expression to convert.
        sympy_variables (list of sympy.Symbol): The symbolic variables in the expression.

    Returns:
        function: A PyTorch function that takes a tensor as input and returns the evaluated expression.
    """
    # Replace SymPy symbols with placeholders for tensor indexing
    pytorch_expression_str = str(sympy_expression)
    for i, var in enumerate(sympy_variables):
        pytorch_expression_str = pytorch_expression_str.replace(str(var), f"x[{i}]")

    # Define a PyTorch function using the string representation of the expression
    def pytorch_function(x):
        # Evaluate the PyTorch expression using eval with safeguards (avoid arbitrary code execution)
        # Consider using a safer approach like creating a symbolic computation graph if security is a concern
        return eval(pytorch_expression_str, {'x': x, 'torch': torch, 'np': torch, 'sqrt':torch.sqrt})

    return pytorch_function

def loss_function(x, z, h_function_pytorch, scenario, clock_penalty_weight=1e-1):
    h_x = h_function_pytorch(x, scenario)  # Calculate predicted measurements
    y = z - h_x
    data_loss = torch.sum(y * y)  # Data loss (sum of squared differences)

    # Identify clock parameters and calculate clock penalty
    clock_params = [
        x[i] for i, param in enumerate(scenario.symbolic_parameter_vector)
        if param.name.startswith("delta_")  # Assuming clock parameters start with "delta_"
    ]
    clock_penalty = clock_penalty_weight * torch.sum(torch.stack(clock_params)**2)  # L2 penalty on clock parameters

    # Total loss = data loss + clock penalty
    total_loss = data_loss + clock_penalty
    return total_loss


# Initialize state estimate and set requires_grad
#print(x_JCLS)
x = torch.tensor(np.array(x_JCL, np.float64), requires_grad=True)

# Define optimizer (e.g., SGD)
optimizer = optim.Adam([x], lr=1e-2)

# Optimization loop
num_epochs = 400
with tqdm(range(num_epochs), desc="Optimizing") as pbar:
    for epoch in pbar:
        optimizer.zero_grad()  # Reset gradients
        loss = loss_function(x, torch.tensor(z), h_function_pytorch, scenario)
        loss.backward()  # Calculate gradients
        optimizer.step()  # Update state estimate (x)
        pbar.set_postfix({"Loss": loss.item()})

        # ... (Logging or printing progress) ...
optimized_state_estimate = x.detach().numpy()
print("")
print("Optimized State Estimate Diff:", 1000*(optimized_state_estimate-scenario.get_true_state()))
print("Final Error", np.linalg.norm(optimized_state_estimate-scenario.get_true_state())*1000,"m")

In [ ]:
import pyproj
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

def ecef_to_latlon(x, y, z):
    """Converts ECEF coordinates to latitude and longitude."""
    ecef = pyproj.Proj(proj='geocent', ellps='WGS84', datum='WGS84')
    lla = pyproj.Proj(proj='latlong', ellps='WGS84', datum='WGS84')
    lon, lat, alt = pyproj.transform(ecef, lla, x, y, z, radians=False)
    return lat, lon

# Example ECEF coordinates
ecef_coords = [[1529.60061385282e3, -4465.33684773589e3, 4275.34001926923e3],
              [1519.60061385282e3, -4475.33684773589e3, 4285.34001926923e3],
              [1509.60061385282e3, -4485.33684773589e3, 4265.34001926923e3]]

# Convert ECEF to latitude/longitude
latitudes, longitudes = [], []
for x, y, z in ecef_coords:
    lat, lon = ecef_to_latlon(x, y, z)
    latitudes.append(lat)
    longitudes.append(lon)

# Create a map projection
projection = ccrs.PlateCarree()

# Create a figure and axes with the map projection
fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': projection})

# Add coastlines and land features
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.LAND)

# Plot the coordinates on the map
ax.scatter(longitudes, latitudes, transform=projection, color='red', marker='o', label='Coordinates')

# Set map extent
ax.set_extent([-180, 180, -90, 90], crs=projection)  # Adjust extent as needed

# Add a title and legend
ax.set_title('Coordinates on Map')
ax.legend()

# Show the plot
plt.show()

In [ ]:
import pyproj
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

def ecef_to_latlon(x, y, z):
    """Converts ECEF coordinates to latitude and longitude."""
    ecef = pyproj.Proj(proj='geocent', ellps='WGS84', datum='WGS84')
    lla = pyproj.Proj(proj='latlong', ellps='WGS84', datum='WGS84')
    lon, lat, alt = pyproj.transform(ecef, lla, x, y, z, radians=False)
    return lat, lon

# Eiffel Tower ECEF coordinates
#[1529.60061385282e3, -4465.33684773589e3, 4275.34001926923e3]
x_eiffel = 1529.60061385282e3#4144612.452
y_eiffel = -4465.33684773589e3#649675.252
z_eiffel = 4275.34001926923e3#4885689.232

# ... (ecef_to_latlon function and Eiffel Tower coordinates) ...

# Create a map projection (Mercator for this example)
projection = ccrs.Mercator()

# Create a figure and axes with the map projection
fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': projection})

# 1. Add more detailed coastlines and land:
ax.add_feature(cfeature.COASTLINE)  # Keep your existing coastline
ax.add_feature(cfeature.LAND, edgecolor='black', facecolor='lightgray')  # Keep existing land

# 2. Add Natural Earth features for higher resolution:
# Land
land_10m = cfeature.NaturalEarthFeature('physical', 'land', '10m',
                                        edgecolor='face', facecolor=cfeature.COLORS['land'])
ax.add_feature(land_10m, zorder=0)  # Add land with lower zorder

# Coastline (higher resolution if available)
coastline_10m = cfeature.NaturalEarthFeature('physical', 'coastline', '10m',
                                            edgecolor='black', facecolor='none')
ax.add_feature(coastline_10m, zorder=1)  # Add coastline with higher zorder

# 3. Add rivers, lakes, and other features:
ax.add_feature(cfeature.RIVERS)
ax.add_feature(cfeature.LAKES)
ax.add_feature(cfeature.BORDERS, linestyle=':')

# 4. Add Natural Earth features for more detail (using 10m scale):
states_provinces = cfeature.NaturalEarthFeature(
    category='cultural',
    name='admin_1_states_provinces_lines',
    scale='10m',
    facecolor='none')
ax.add_feature(states_provinces, edgecolor='gray', zorder=2)

# Ocean bathymetry (if desired):
ocean_10m = cfeature.NaturalEarthFeature('physical', 'ocean', '10m',
                                         edgecolor='face', facecolor=cfeature.COLORS['water'])
ax.add_feature(ocean_10m, zorder=-1)  # Add ocean with lower zorder

# ... (Add other features with '10m' or other scales) ...

# 5. Plot the Eiffel Tower coordinates:
lat_eiffel, lon_eiffel = ecef_to_latlon(x_eiffel, y_eiffel, z_eiffel)
ax.scatter(lon_eiffel, lat_eiffel, transform=ccrs.PlateCarree(),
           color='red', marker='o', label='Eiffel Tower', zorder=10)  # zorder ensures it's on top

# Set map extent
ax.set_extent([lon_eiffel - .2, lon_eiffel + .2,
               lat_eiffel - .2, lat_eiffel + .2], crs=ccrs.PlateCarree())

# Add a title and legend
ax.set_title('Local Map Overlay - Eiffel Tower')
ax.legend()

# Show the plot
plt.show()

##### Antiquated (heatmaps)

In [ ]:
print(min(map_position_errors.flatten()),max(map_position_errors.flatten()))
print(np.unravel_index(np.argmax(map_position_errors), map_position_errors.shape))
r1 = map_position_errors
#r1[6,0] = r1[6,2]
fitted_plane_matrix = gaussian_filter(map_position_errors, sigma=.7)#fit_plane_to_matrix_values(r1)
print(min(r1.flatten()),max(r1.flatten()))

#display_heatmap(il_position_errors, title="IL Position Errors (m)", xlabel="Number of Satellites", x_range=num_satellites_range, ylabel="Number of Users", y_range=num_users_range)
#display_heatmap(lm_position_errors, title="LM Position Errors (m)", xlabel="Number of Satellites", x_range=num_satellites_range, ylabel="Number of Users", y_range=num_users_range)
r1 = map_position_errors
sp.pprint(sp.Matrix(r1))
#r1[3,0] = r1[2,0]
r2 = r1-il_position_errors
#r2[3,2] = r1[3,3]
loc_errors_mat = gaussian_filter(r1, sigma=.3)
display_heatmap(loc_errors_mat, title="Mean JCLS UE Position Error after $.5$s (m)", xlabel="$N_\mathrm{Sat}$", x_range=num_satellites_range, ylabel="$N_\mathrm{UE}$", y_range=num_users_range, log=True, zlabel="m")
print(r1)

loc_errors_mat = gaussian_filter(map_sync_errors, sigma=.75)
#loc_errors_mat[3,2] = loc_errors_mat[3,3]
#loc_errors_mat[1,1] = loc_errors_mat[1,0]
display_heatmap(loc_errors_mat*1e9, title="Mean JCLS Synchronization Error after $.5$s (ns)", xlabel="$N_\mathrm{Sat}$", x_range=num_satellites_range, ylabel="$N_\mathrm{UE}$", y_range=num_users_range, zlabel="ns")

#display_heatmap(r1, title="JCLS Position Errors after $.5$s (m)", xlabel="$N_\mathrm{NTN}$", x_range=num_satellites_range, ylabel="$N_\mathrm{UE}$", y_range=num_users_range)
#display_heatmap(r2, title="JCLS Improvement (m)", xlabel="$N_\mathrm{NTN}$", x_range=num_satellites_range, ylabel="$N_\mathrm{UE}$", y_range=num_users_range)

##### Test 4: Fisher Information DL/SL bandwidth (Failed)

In [ ]:
def build_sigma_matrix_from_snr(scenario, snr_dl, snr_sl):
    """
    Builds a new Sigma matrix based on the given SNR values for downlinks (DL) and sidelinks (SL).
    Uses scenario information to determine link types and calculate variances.

    Args:
        scenario (Scenario): The scenario object.
        snr_dl (float): The desired SNR value for downlinks (in linear scale).
        snr_sl (float): The desired SNR value for sidelinks (in linear scale).

    Returns:
        np.ndarray: The new Sigma matrix (covariance matrix).
    """

    links = scenario.get_links()
    num_links = len(links)
    variances = np.zeros(num_links)  # Initialize variances array

    for i, link in enumerate(links):
        if isinstance(link.transmitter, Satellite):  # Check if it's a downlink
            snr = snr_dl  # Use downlink SNR
            # Avoid division by zero when snr_dl is 0 by using a small positive value
            if snr_dl == 0:
                snr_for_calculation = 1e-10  # Use a small positive value instead of 0
            else:
                snr_for_calculation = snr_dl
        else:  # Otherwise, it's a sidelink
            snr = snr_sl  # Use sidelink SNR
            snr_for_calculation = snr_sl

        # Calculate variance based on SNR and link properties
        sigma_rician = 3e8 / 1000 * np.sqrt(1 / (8 * (np.pi * link.transmitter.tx_bw)**2 * snr_for_calculation))
        variance = sigma_rician**2  # Assuming Rician channel

        variances[i] = variance  # Store the variance for this link

    # Create the Sigma matrix (diagonal covariance matrix)
    Sigma = np.diag(variances)
    return Sigma

def build_sigma_matrix_from_bw(scenario, dl_bw, sl_bw):
    """
    Builds a new Sigma matrix based on the given BW values for downlinks (DL) and sidelinks (SL).
    Uses scenario information to determine link types and calculate variances.

    Args:
        scenario (Scenario): The scenario object.
        snr_dl (float): The desired BW value for downlinks (in linear scale).
        snr_sl (float): The desired BW value for sidelinks (in linear scale).

    Returns:
        np.ndarray: The new Sigma matrix (covariance matrix).
    """

    links = scenario.get_links()
    num_links = len(links)
    variances = np.zeros(num_links)  # Initialize variances array

    for i, link in enumerate(links):
        if isinstance(link.transmitter, Satellite):  # Check if it's a downlink
            bw = dl_bw  # Use downlink SNR
        else:  # Otherwise, it's a sidelink
            bw = sl_bw

        # Calculate variance based on SNR and link properties
        sigma_rician = 3e8 / 1000 * np.sqrt(1 / (8 * (np.pi * bw)**2 * link.get_snr_abs()))
        variance = sigma_rician**2  # Assuming Rician channel

        variances[i] = variance  # Store the variance for this link

    # Create the Sigma matrix (diagonal covariance matrix)
    Sigma = np.diag(variances)
    return Sigma

def psd(matrix):
    try:
        _ = np.linalg.cholesky(matrix + 1e-10 * np.eye(matrix.shape[0])) #small value for psd check
        return True
    except np.linalg.LinAlgError:
        return False

import numpy as np

def remove_dependent_measurements(J, h=None, Sigma=None, tol=1e-10):
    """
    Removes linearly dependent rows from the Jacobian (J) and
    corresponding entries from the measurement vector (h) and
    covariance matrix (Sigma).

    Args:
        J (np.ndarray): The Jacobian matrix.
        h (np.ndarray, optional): The measurement vector. Defaults to None.
        Sigma (np.ndarray, optional): The measurement covariance matrix. Defaults to None.
        tol (float, optional): Tolerance for determining linear dependence. Defaults to 1e-10.

    Returns:
        tuple: A tuple containing the updated J, h, and Sigma (if provided).
    """

    # 1. Identify dependent rows using QR decomposition
    Q, R = np.linalg.qr(J)
    dependent_row_indices = np.where(np.abs(np.diag(R)) < tol)[0]

    # 2. Remove dependent rows from Jacobian
    J_independent = np.delete(J, dependent_row_indices, axis=0)
    h_independent = None
    Sigma_independent = None

    if h is not None:
        h_independent = np.delete(h, dependent_row_indices, axis=0)

    if Sigma is not None:
        Sigma_independent = np.delete(np.diag(Sigma), dependent_row_indices)
        Sigma_independent = np.diag(Sigma_independent)

    return J_independent, h_independent, Sigma_independent

def generate_CRLB_BW(dl_bw, sl_bw):
    """
    Generates data for heatmaps of JCLS accuracy.

    Args:
        num_satellites_range (range): Range of number of satellites to test.
        num_users_range (range): Range of number of users to test.
        num_iterations (int, optional): Number of iterations for each algorithm. Defaults to 25.

    Returns:
        tuple: A tuple containing the following:
            - il_position_errors (np.ndarray): Heatmap of average position errors for IL.
            - il_sync_errors (np.ndarray): Heatmap of average sync errors for IL.
            - lm_position_errors (np.ndarray): Heatmap of average position errors for LM.
            - lm_sync_errors (np.ndarray): Heatmap of average sync errors for LM.
            - map_position_errors (np.ndarray): Heatmap of average position errors for MAP.
            - map_sync_errors (np.ndarray): Heatmap of average sync errors for MAP.
    """

    # Initialize heatmap arrays
    loc_mat = np.zeros((len(dl_bw), len(sl_bw)))
    sync_mat = np.zeros((len(dl_bw), len(sl_bw)))
    crlb_mat = np.zeros((len(dl_bw), len(sl_bw)))


    # Iterate through satellite and user configurations
    total_iterations = len(dl_bw) * len(sl_bw)
    #print(f"Generating FIM  data for {total_iterations} configurations...")
    progress_bar = tqdm(total=total_iterations, desc="Generating Heatmap Data")
    for i, dl in enumerate(dl_bw):
        for j, sl in enumerate(sl_bw):
            # Create a new scenario for each test case
            scenario = Scenario(num_users=3, num_satellites=3, clock_std_dev_seconds=1e-6)

            Sigma =  build_sigma_matrix_from_bw(scenario, dl, sl)
            Sigma = scenario.get_measurement_covariance()
            J = scenario.evaluate_jacobian(scenario.get_true_state())
            J_ind, _, Sigma_ind = remove_dependent_measurements(J, Sigma=Sigma)
            FIM = J_ind.T @ np.linalg.inv(Sigma_ind) @ J_ind
            FIM = J.T @ np.linalg.inv(Sigma) @ J
            #print(np.linalg.cond(FIM))
            #print(psd(Sigma_ind), psd(FIM))

            clock_indices = np.array([i for i, param in enumerate(scenario.symbolic_parameter_vector)
                                      if param.name.startswith('delta_')],
                                      dtype=np.int64)
            position_indices = np.array([i for i, param in enumerate(scenario.symbolic_parameter_vector)
                                         if param.name.startswith(('x_', 'y_', 'z_'))],
                                         dtype=np.int64)

            J_x_no_clock = np.delete(J_ind, clock_indices, axis=1)
            FIM_loc = J_x_no_clock.T @ np.linalg.inv(Sigma_ind) @ J_x_no_clock

            J_clock = np.delete(J_ind, position_indices, axis=1)
            FIM_clock = J_clock.T @ np.linalg.inv(Sigma_ind) @ J_clock

            loc_mat[i,j] = np.trace(np.linalg.inv(FIM_loc)) / scenario.num_users#np.linalg.norm(FIM_loc)
            sync_mat[i,j] =  np.trace(np.linalg.pinv(FIM_clock)) / (scenario.num_users+scenario.num_satellites)

            progress_bar.set_description(f"DL BW: {dl}, SL BW: {sl}")
            progress_bar.update(1)

    progress_bar.close()  # Close the progress bar when done
    return loc_mat, sync_mat, #crlb_mat

#num_satellites_range = [5, 7, 9]#range(3, 14+1)
#num_users_range = [2, 3] #range(1, 7+1)
dl = np.linspace(20e6,50e6,2)
sl = np.linspace(1e6,1000e6,2)
loc, sync = generate_CRLB_BW(dl,sl)
clrb= loc + sync
print(loc)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np

crlb1 = loc
display_heatmap(loc*1000, title="Localization CRLB 0dB 0dB", xlabel="DL BW", x_range=dl, ylabel="SL BW", y_range=sl, zlabel="$\,$m",log=False)
#plot_heatmap(loc*1000, title="Localization CRLB for static JCLS (m)", xlabel="$N_\mathrm{NTN}$", ylabel="$N_\mathrm{UE}$", xticks=num_satellites_range, yticks=num_users_range)
#display_heatmap(sy, title="Localization CRLB for static JCLS (km)", xlabel="$N_\mathrm{NTN}$", x_range=num_satellites_range, ylabel="$N_\mathrm{UE}$", y_range=num_users_range)
display_heatmap(sync*1000/3e8, title="Synchronization CRLB 0dB 0dB", xlabel="DL BW", x_range=dl, ylabel="SL BW", y_range=sl, zlabel="$\,$s")

##### Test 3: User Speed (Failed)

In [ ]:
from tqdm import tqdm
import numpy as np

def generate_data_for_user_speed(user_speeds, num_iterations=25, num_users=3, num_satellites=10):
    """
    Generates data for analyzing JCLS accuracy with different user speeds.

    Args:
        user_speeds (list or np.ndarray): A list of user speeds to test (in meters per iteration).
        num_iterations (int, optional): Number of iterations for each algorithm. Defaults to 25.
        num_users (int, optional): Number of users in the scenario. Defaults to 3.
        num_satellites (int, optional): Number of satellites in the scenario. Defaults to 10.

    Returns:
        tuple: A tuple containing the following:
            - il_position_errors (np.ndarray): Average position errors for IL.
            - il_sync_errors (np.ndarray): Average sync errors for IL.
            - lm_position_errors (np.ndarray): Average position errors for LM.
            - lm_sync_errors (np.ndarray): Average sync errors for LM.
            - map_position_errors (np.ndarray): Average position errors for MAP.
            - map_sync_errors (np.ndarray): Average sync errors for MAP.
    """

    # Initialize arrays to store results
    il_position_errors = []
    il_sync_errors = []
    lm_position_errors = []
    lm_sync_errors = []
    map_position_errors = []
    map_sync_errors = []

    # Iterate through user speeds
    for speed in tqdm(user_speeds, desc="Generating Data for User Speed"):
        print(f"Generating data for user speed: {speed}")
        # Set the User.speed class variable
        User.speed = speed

        # Create a new scenario
        scenario = Scenario(num_users=num_users, num_satellites=num_satellites, clock_std_dev_seconds=1e-6)

        # Initialize optimizer and get measurements
        optimizer = Optimizer()
        x_init = optimizer.initialize_state(scenario, error_range=100)  # Initialize with large error range (100 km)
        z = scenario.query_measurements()

        # Run IL and store results
        x_il = optimizer.run(algorithm="IL", scenario=scenario, x=x_init, z=z, num_steps=15, tol=1e-8, verbose=False)
        il_position_errors.append(optimizer.calculate_average_position_error(scenario, x_il))
        il_sync_errors.append(optimizer.calculate_average_clock_error(scenario, x_il))

        # Run LM with IL estimate as input and store results
        try:
            x_lm = optimizer.run(algorithm="LM", scenario=scenario, x=x_il, z=z, num_steps=20, verbose=False)
        except:
            x_lm = x_il
        lm_position_errors.append(optimizer.calculate_average_position_error(scenario, x_lm))
        lm_sync_errors.append(optimizer.calculate_average_clock_error(scenario, x_lm))

        # Run MAP filtering with LM estimate as input and store results
        x_map = x_lm.copy()
        P = optimizer.calculate_state_covariance(scenario, x_lm) / 1.1
        for _ in range(num_iterations):
            z = scenario.query_measurements()
            try:
                P, x_map = optimizer.map_filter_iteration(scenario, P, x_map, z, verbose=False)
            except:
                P, x_map = map_filter_iteration(None, scenario, P, x_map, z, verbose=False)
            scenario.move_nodes()

        map_position_errors.append(optimizer.calculate_average_position_error(scenario, x_map))
        map_sync_errors.append(optimizer.calculate_average_clock_error(scenario, x_map))


    # Convert results to NumPy arrays
    il_position_errors = np.array(il_position_errors)
    il_sync_errors = np.array(il_sync_errors)
    lm_position_errors = np.array(lm_position_errors)
    lm_sync_errors = np.array(lm_sync_errors)
    map_position_errors = np.array(map_position_errors)
    map_sync_errors = np.array(map_sync_errors)

    return il_position_errors, il_sync_errors, lm_position_errors, lm_sync_errors, map_position_errors, map_sync_errors

user_speeds = np.linspace(0,50,10)#np.logspace(-100,np.log10(50),10)
results_speed = generate_data_for_user_speed(user_speeds, num_iterations=25, num_users=3, num_satellites=10)
il_pos_speed, il_sync_speed, lm_pos_speed, lm_sync_speed, map_pos_speed, map_sync_speed= results_speed

In [ ]:
ys = [il_pos_speed, lm_pos_speed, map_pos_speed]
#ys_fitted = gaussian_filter(ys, sigma=.25)#[fit_and_resample_power_law(clock_std_devs, y, clock_std_devs) for y in ys]
labels = ["Conventional TOA", "Coarse JCLS", "Refined JCLS ($.5\,$s)"]
ieee_flexible_plot(user_speeds, ys, labels,
                        xlabel=r'$\sigma_\delta$ [s]', ylabel=r'Average UE error [m]', title="pos error vary clock",
                        plot_type='line',  # 'line' or 'scatter'
                        markers=None, linestyles=None,
                        save_path=None, figsize=(3.5, 3),
                        log_y=True,
                        log_x=False,
                        legend_loc="best")

#### Save Workspace

In [ ]:
save_workspace('/content/drive/MyDrive/my_workspace.pkl')

Workspace saved to '/content/drive/MyDrive/my_workspace.pkl'.


### Load Workspace

In [ ]:
load_workspace('/content/drive/MyDrive/my_workspace.pkl')

Workspace loaded from '/content/drive/MyDrive/my_workspace.pkl'.


In [ ]:
def download_tle_if_needed(
    url: str,
    cache_path: Path,
    force_download: bool = False,
) -> None:
    """
    Download Starlink TLEs unless a cached file exists.

    Robust behavior:
      1. If a cache exists and force_download=False, use it.
      2. If download fails but cache exists, use the cache.
      3. Try several CelesTrak URL variants.
      4. Raise a clear error only if no download succeeds and no cache exists.
    """
    if cache_path.exists() and not force_download:
        print(f"Using cached TLE file: {cache_path}")
        return

    candidate_urls = [
        url,
        "https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle",
        "https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=2le",
        "https://celestrak.org/NORAD/elements/starlink.txt",
        "https://celestrak.org/NORAD/elements/supplemental/starlink.txt",
    ]

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/121.0 Safari/537.36"
        ),
        "Accept": "text/plain,text/html,*/*",
        "Referer": "https://celestrak.org/",
    }

    last_error = None

    for candidate_url in candidate_urls:
        try:
            print(f"Attempting TLE download from:\n  {candidate_url}")

            req = urllib.request.Request(candidate_url, headers=headers)

            with urllib.request.urlopen(req, timeout=60) as response:
                tle_text = response.read().decode("utf-8", errors="replace")

            tle_text = tle_text.strip()

            if (
                len(tle_text) == 0
                or "No GP data found" in tle_text
                or "<html" in tle_text.lower()
                or "forbidden" in tle_text.lower()
            ):
                raise RuntimeError("Downloaded content does not appear to be a valid TLE file.")

            # Basic sanity check: TLE files should contain line 1 / line 2 records.
            if "\n1 " not in "\n" + tle_text or "\n2 " not in "\n" + tle_text:
                raise RuntimeError("Downloaded text does not contain recognizable TLE line records.")

            cache_path.parent.mkdir(exist_ok=True)
            cache_path.write_text(tle_text + "\n", encoding="utf-8")

            print(f"Saved TLE file to: {cache_path}")
            return

        except Exception as err:
            last_error = err
            print(f"Download attempt failed: {repr(err)}")

    if cache_path.exists():
        print(
            f"All download attempts failed, but a cached TLE file exists. "
            f"Using cached file: {cache_path}"
        )
        return

    # Try to find the old cache from the previous script.
    old_cache_candidates = [
        Path("starlink_visibility_outputs/starlink_current.tle"),
        Path("../starlink_visibility_outputs/starlink_current.tle"),
    ]

    for old_cache in old_cache_candidates:
        if old_cache.exists():
            cache_path.parent.mkdir(exist_ok=True)
            shutil.copy(old_cache, cache_path)
            print(f"Copied old cached TLE file:\n  {old_cache}\n  -> {cache_path}")
            return

    raise RuntimeError(
        "Could not download Starlink TLEs and no cached TLE file was found. "
        "This is likely a temporary CelesTrak 403/rate-limit/cloud-IP issue. "
        "Workaround: download the Starlink TLE file manually in a browser and save it as:\n"
        f"  {cache_path}\n"
        f"Last error was: {repr(last_error)}"
    )